# Construcción del dataset analítico para predicción de reapertura de bugs

Este notebook construye el dataset analítico utilizado para estudiar la reapertura de bugs en proyectos de software gestionados en Jira.

El objetivo es generar una tabla con una fila por bug, utilizando la información disponible hasta el momento de su primera resolución. La variable objetivo (`Reopened_Target`) indica si el bug vuelve posteriormente a un estado activo o explícitamente reabierto.

La construcción del dataset se realiza a partir de las tablas originales del dataset TAWOS y de tablas auxiliares que documentan la selección de proyectos y la clasificación de estados.

## 1. Configuración inicial

Se importan las librerías necesarias, se configuran las rutas de trabajo y se establece la conexión con la base MySQL que contiene los datos.

In [1]:
import ast
import os
import re
from collections import defaultdict, deque

from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [2]:
BASE_DIR = Path("/app")
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

load_dotenv(BASE_DIR / ".env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", "3306"))
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

url = URL.create(
    drivername="mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(url)

with engine.connect() as conn:
    db_actual = conn.execute(text("SELECT DATABASE();")).scalar()

print("Base conectada:", db_actual)

Base conectada: tawos


## 2. Fuentes de datos

Se utilizan las tablas originales del dataset TAWOS, que contienen información de issues, historial de cambios, comentarios, versiones, componentes, usuarios, proyectos y relaciones entre issues.

Además, se utilizan tres tablas auxiliares:

- `analysis_project_scope`: define los proyectos incluidos en el análisis.
- `status_group`: define los grupos funcionales de estados.
- `status_map_project`: asigna cada estado observado en cada proyecto a un grupo funcional.

Estas tablas permiten transformar los estados específicos de cada proyecto en categorías comparables para construir la variable objetivo.

In [3]:
required_tables = [
    "project",
    "repository",
    "sprint",
    "version",
    "affected_version",
    "fix_version",
    "issue",
    "issue_link",
    "user",
    "comment",
    "change_log",
    "issue_component",
    "component",
    "analysis_project_scope",
    "status_group",
    "status_map_project",
]

tables_df = pd.read_sql(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = DATABASE()
    """,
    engine
)

available_tables = set(tables_df["TABLE_NAME"].tolist())
missing_tables = [t for t in required_tables if t not in available_tables]

missing_tables

[]

## 3. Tamaño de las tablas utilizadas

Se revisa la cantidad de registros de cada tabla para documentar el volumen inicial de datos.

In [4]:
row_counts = []

for table in required_tables:
    query = f"SELECT COUNT(*) AS n FROM `{table}`"
    n = pd.read_sql(query, engine)["n"].iloc[0]
    row_counts.append({"table": table, "rows": n})

row_counts_df = pd.DataFrame(row_counts).sort_values("table")
row_counts_df

,table,rows
4,affected_version,264004
13,analysis_project_scope,8
10,change_log,9253419
9,comment,1518327
12,component,2001
5,fix_version,224254
6,issue,458232
11,issue_component,366922
7,issue_link,246587
0,project,39


## 4. Carga de tablas auxiliares

Se cargan las tablas que definen el alcance del análisis y la clasificación de estados de Jira.

La tabla `analysis_project_scope` contiene los proyectos seleccionados.  
La tabla `status_group` define los grupos funcionales de estados.  
La tabla `status_map_project` indica a qué grupo pertenece cada estado observado dentro de cada proyecto.

In [5]:
project_scope = pd.read_sql(
    """
    SELECT *
    FROM analysis_project_scope
    ORDER BY Project_ID
    """,
    engine
)

status_group = pd.read_sql(
    """
    SELECT *
    FROM status_group
    ORDER BY ID
    """,
    engine
)

status_map = pd.read_sql(
    """
    SELECT *
    FROM status_map_project
    ORDER BY Project_ID, Status_Name
    """,
    engine
)

print("project_scope:", project_scope.shape)
print("status_group:", status_group.shape)
print("status_map:", status_map.shape)

project_scope: (8, 2)
status_group: (6, 8)
status_map: (716, 7)


In [6]:
display(project_scope)
display(status_group)
display(status_map.head(20))

,Project_ID,Notes
0,7,Seleccionado por alta señal preliminar de reap...
1,8,Seleccionado por señal preliminar de reapertur...
2,9,Seleccionado por señal preliminar de reapertur...
3,12,"Seleccionado por alto volumen, señal prelimina..."
4,13,Seleccionado por alta señal preliminar de reap...
5,34,Seleccionado por alta señal preliminar de reap...
6,36,Seleccionado por señal preliminar de reapertur...
7,42,Seleccionado por alta señal preliminar de reap...


,ID,Code,Name,Description,Is_Resolution_Like,Is_Fix_Resolution,Is_Reopen_Destination,Is_Usable_For_Target
0,1,active,Activo,"El ticket sigue en trabajo, revisión, testing,...",0,0,1,1
1,2,resolution,Resolución/cierre,El ticket llegó a un cierre o resolución usabl...,1,1,0,1
2,3,non_fix_resolution,Cierre no asociado a corrección,El ticket fue cerrado por una razón no asociad...,1,0,0,1
3,4,explicit_reopen,Reapertura explícita,"Estado explícito de reapertura, por ejemplo Re...",0,0,1,1
4,5,unclear,Dudoso,Estado ambiguo o no clasificado. No se usa par...,0,0,0,0
5,6,validation,Validación/revisión/testing,"El ticket se encuentra en revisión, QA, testin...",0,0,0,1


,ID,Project_ID,Status_Name,Status_Key,Appearances,Status_Group_ID,Notes
0,4,1,Done,11a6767d5674c7e45f7e00dc525762275b3a48491ad604...,6352,2,NaN
1,3,1,In PR,2e66f5de0e26662c16188a22e6186d21dc2dc38e1d5448...,1352,1,NaN
2,2,1,In Progress,b4cc4b07c300103a4bc38e01e084cc945144a7eb0ae61d...,4578,1,NaN
3,5,1,Open,ed077f3d8125d60dca1979c7133601bd187d47c73ed997...,24,1,NaN
4,1,1,To Do,150d92c40cfd0d1e52553e8c938ab117d0b1ee048bd1b6...,4149,1,NaN
5,8,2,Closed,c21ead0614e7e1b70edb3759e1d19ca1e12e6036980d10...,1460,2,NaN
6,15,2,Done,11a6767d5674c7e45f7e00dc525762275b3a48491ad604...,356,2,NaN
7,10,2,In Progress,b4cc4b07c300103a4bc38e01e084cc945144a7eb0ae61d...,884,1,NaN
8,11,2,Investigating,13e3869dfd816830d65591c794719556dbd34d418da4c0...,140,1,NaN
9,6,2,Open,ed077f3d8125d60dca1979c7133601bd187d47c73ed997...,723,1,NaN


### Proyectos incluidos

A partir de `analysis_project_scope` se obtiene la lista de proyectos incluidos en el análisis.

In [7]:
PROJECT_IDS = (
    project_scope["Project_ID"]
    .dropna()
    .astype(int)
    .sort_values()
    .tolist()
)

PROJECT_IDS

[7, 8, 9, 12, 13, 34, 36, 42]

### Validación de clasificación de estados

Se verifica que todos los proyectos incluidos tengan estados clasificados y que todos los grupos usados en `status_map_project` estén definidos en `status_group`.

In [8]:
projects_in_scope = set(PROJECT_IDS)
projects_in_status_map = set(status_map["Project_ID"].dropna().astype(int).unique())

missing_projects_in_status_map = sorted(projects_in_scope - projects_in_status_map)

missing_projects_in_status_map

[]

In [9]:
valid_status_group_ids = set(status_group["ID"].dropna().astype(int).unique())
used_status_group_ids = set(status_map["Status_Group_ID"].dropna().astype(int).unique())

invalid_status_group_ids = sorted(used_status_group_ids - valid_status_group_ids)

invalid_status_group_ids

[]

In [10]:
status_map_by_project = (
    status_map
    .groupby("Project_ID")
    .size()
    .reset_index(name="n_statuses")
    .sort_values("Project_ID")
)

status_map_by_project

,Project_ID,n_statuses
0,1,5
1,2,10
2,3,22
3,4,9
4,5,6
5,6,4
6,7,8
7,8,7
8,9,7
9,10,6


### Unión entre estados y grupos funcionales

Se incorpora a `status_map_project` la información de `status_group`, de modo de contar con una tabla única que indique el grupo funcional y las banderas metodológicas asociadas a cada estado.

In [11]:
status_map_full = status_map.merge(
    status_group,
    left_on="Status_Group_ID",
    right_on="ID",
    how="left",
    suffixes=("", "_group")
)

status_map_full = status_map_full[
    [
        "Project_ID",
        "Status_Name",
        "Status_Group_ID",
        "Code",
        "Name",
        "Is_Resolution_Like",
        "Is_Fix_Resolution",
        "Is_Reopen_Destination",
        "Is_Usable_For_Target",
    ]
].rename(columns={
    "Code": "Status_Group_Code",
    "Name": "Status_Group_Name",
})

status_map_full.head(20)

,Project_ID,Status_Name,Status_Group_ID,Status_Group_Code,Status_Group_Name,Is_Resolution_Like,Is_Fix_Resolution,Is_Reopen_Destination,Is_Usable_For_Target
0,1,Done,2,resolution,Resolución/cierre,1,1,0,1
1,1,In PR,1,active,Activo,0,0,1,1
2,1,In Progress,1,active,Activo,0,0,1,1
3,1,Open,1,active,Activo,0,0,1,1
4,1,To Do,1,active,Activo,0,0,1,1
5,2,Closed,2,resolution,Resolución/cierre,1,1,0,1
6,2,Done,2,resolution,Resolución/cierre,1,1,0,1
7,2,In Progress,1,active,Activo,0,0,1,1
8,2,Investigating,1,active,Activo,0,0,1,1
9,2,Open,1,active,Activo,0,0,1,1


In [12]:
status_map_full.isna().sum()

Project_ID               0
Status_Name              0
Status_Group_ID          0
Status_Group_Code        0
Status_Group_Name        0
Is_Resolution_Like       0
Is_Fix_Resolution        0
Is_Reopen_Destination    0
Is_Usable_For_Target     0
dtype: int64

## 5. Carga de bugs incluidos en el análisis

Se cargan los issues de tipo `Bug` correspondientes a los proyectos seleccionados. Esta tabla será la base del dataset analítico, donde cada fila final representará un bug.

In [13]:
project_ids_sql = ",".join(str(p) for p in PROJECT_IDS)

issues = pd.read_sql(
    f"""
    SELECT
        ID AS Issue_ID,
        Jira_ID,
        Project_ID,
        Issue_Key,
        Type,
        Priority,
        Status,
        Resolution,
        Creation_Date,
        Resolution_Date,
        Last_Updated,
        Creator_ID,
        Reporter_ID,
        Assignee_ID,
        Title,
        Description,
        Description_Text,
        Description_Code,
        Sprint_ID,
        Story_Point,
        Timespent,
        In_Progress_Minutes,
        Total_Effort_Minutes,
        Resolution_Time_Minutes,
        Pull_Request_URL
    FROM `issue`
    WHERE Type = 'Bug'
      AND Project_ID IN ({project_ids_sql})
    """,
    engine,
    parse_dates=[
        "Creation_Date",
        "Resolution_Date",
        "Last_Updated",
    ]
)

issues.shape

(80489, 25)

In [14]:
# Fecha final observable de cada proyecto.
# Se toma el último evento registrado en change_log para cualquier issue
# del proyecto, no solamente para los bugs. La consulta devuelve una fila
# por proyecto y no representa un problema de memoria.
project_observation_dates = pd.read_sql(
    f"""
    SELECT
        i.Project_ID,
        MAX(cl.Creation_Date) AS Project_Observation_End_Date
    FROM change_log cl
    JOIN `issue` i
        ON i.ID = cl.Issue_ID
    WHERE i.Project_ID IN ({project_ids_sql})
    GROUP BY i.Project_ID
    ORDER BY i.Project_ID
    """,
    engine,
    parse_dates=["Project_Observation_End_Date"],
)

if project_observation_dates["Project_ID"].duplicated().any():
    raise ValueError(
        "La consulta devolvió más de una fecha final por proyecto."
    )

missing_project_end_dates = sorted(
    set(PROJECT_IDS)
    - set(
        project_observation_dates["Project_ID"]
        .dropna()
        .astype(int)
    )
)

if missing_project_end_dates:
    raise ValueError(
        "No se obtuvo fecha final de observación para estos proyectos: "
        f"{missing_project_end_dates}"
    )

display(project_observation_dates)


,Project_ID,Project_Observation_End_Date
0,7,2020-10-08 10:08:58
1,8,2020-09-04 12:47:54
2,9,2020-10-20 17:21:19
3,12,2020-10-24 13:45:51
4,13,2020-09-09 17:06:15
5,34,2020-10-22 13:38:02
6,36,2020-10-22 17:14:18
7,42,2018-11-13 17:20:40


In [15]:
issues_by_project = (
    issues
    .groupby("Project_ID")
    .size()
    .reset_index(name="n_bugs")
    .sort_values("Project_ID")
)

issues_by_project

,Project_ID,n_bugs
0,7,646
1,8,6152
2,9,399
3,12,15742
4,13,3455
5,34,41355
6,36,5421
7,42,7319


In [16]:
issues.head()

,Issue_ID,Jira_ID,Project_ID,Issue_Key,Type,Priority,Status,Resolution,Creation_Date,Resolution_Date,Last_Updated,Creator_ID,Reporter_ID,Assignee_ID,Title,Description,Description_Text,Description_Code,Sprint_ID,Story_Point,Timespent,In_Progress_Minutes,Total_Effort_Minutes,Resolution_Time_Minutes,Pull_Request_URL
0,27388,175552,7,ALOY-1740,Bug,Critical,Open,NaN,2020-09-19 18:13:16,NaT,2020-09-27 17:00:21,4899.0,4899.0,4898.0,"""Build times impact by assets & lib files in w...","""*Issue details:* This issue is about app bui...","""""""*Issue details:* This issue is about app b...",,NaN,NaN,NaN,0.0,0.0,0.0,None
1,27390,175488,7,ALOY-1738,Bug,Critical,Closed,Fixed,2020-08-28 08:56:26,2020-09-14 19:27:05,2020-09-14 19:27:05,4901.0,4901.0,4897.0,"""CLI 8.1.0: crash when to listening backbone.e...","""When you add a backbone event listener to a r...","""""""When you add a backbone event listener to a...",""" function doSomething(e) { console.log('d...",NaN,NaN,NaN,70.0,24783.0,25110.0,None
2,27392,175421,7,ALOY-1736,Bug,None,Closed,Fixed,2020-08-07 15:26:00,2020-08-13 15:39:55,2020-08-13 15:39:55,4903.0,4903.0,4897.0,"""aloy new fails if dev dependencies do not exi...","""If dev dependencies are missing or do not exi...","""""""If dev dependencies are missing or do not e...",""" [INFO] Project created successfully in 19s...",NaN,NaN,NaN,0.0,8607.0,8653.0,None
3,27394,175387,7,ALOY-1734,Bug,High,Closed,Fixed,2020-07-30 21:48:49,2020-08-11 21:24:48,2020-08-11 21:24:48,4904.0,4904.0,4897.0,"""JS files imported in """"alloy.js"""" don't have ...","""*Summary:* As of Titanium 9.0.0, a JS files ...","""""""*Summary:* As of Titanium 9.0.0, a JS file...",""" console.log(""""""""@@@ Alloy = """""""" + (typeof ...",571.0,NaN,NaN,54.0,9162.0,17255.0,None
4,27397,175065,7,ALOY-1731,Bug,None,Open,NaN,2020-06-02 19:12:03,NaT,2020-06-02 19:12:03,4906.0,4906.0,4906.0,"""Webpack: Alloy loader recompiles unchanged co...","""The Alloy loader uses [addDependency|https://...","""""""The Alloy loader uses [addDependency|https:...",,NaN,NaN,NaN,0.0,0.0,0.0,None


## 6. Historial de cambios de estado

Para construir la primera resolución y la reapertura posterior se utiliza el historial de cambios de estado registrado en `change_log`.

Se consideran únicamente cambios donde `Field = 'status'`, correspondientes a bugs de los proyectos incluidos.

In [17]:
status_changes = pd.read_sql(
    f"""
    SELECT
        cl.ID AS Change_ID,
        cl.Issue_ID,
        i.Project_ID,
        cl.Creation_Date AS Change_Date,
        cl.From_String AS From_Status,
        cl.To_String AS To_Status,
        cl.Author_ID
    FROM change_log cl
    JOIN `issue` i
        ON i.ID = cl.Issue_ID
    WHERE i.Type = 'Bug'
      AND i.Project_ID IN ({project_ids_sql})
      AND cl.Field = 'status'
    """,
    engine,
    parse_dates=["Change_Date"]
)

status_changes.shape

(271170, 7)

In [18]:
status_changes_by_project = (
    status_changes
    .groupby("Project_ID")
    .size()
    .reset_index(name="n_status_changes")
    .sort_values("Project_ID")
)

status_changes_by_project

,Project_ID,n_status_changes
0,7,1979
1,8,13621
2,9,1197
3,12,51988
4,13,12539
5,34,159574
6,36,10775
7,42,19497


In [19]:
status_changes.head()

,Change_ID,Issue_ID,Project_ID,Change_Date,From_Status,To_Status,Author_ID
0,392428,27390,7,2020-08-28 14:23:44,Open,In Progress,4897.0
1,392430,27390,7,2020-08-28 15:33:51,In Progress,In Review,4897.0
2,392432,27390,7,2020-09-14 17:14:28,In Review,In QE Test,4897.0
3,392433,27390,7,2020-09-14 19:27:02,In QE Test,Resolved,4902.0
4,392434,27390,7,2020-09-14 19:27:05,Resolved,Closed,4902.0


### Clasificación funcional de los estados

Cada estado de Jira se asocia a un grupo funcional mediante `status_map_project` y `status_group`.

Esta clasificación permite identificar qué transiciones corresponden a resolución, reapertura explícita, retorno a estado activo o estados de validación.

In [20]:
to_status_map = status_map_full[
    [
        "Project_ID",
        "Status_Name",
        "Status_Group_Code",
        "Is_Resolution_Like",
        "Is_Fix_Resolution",
        "Is_Reopen_Destination",
        "Is_Usable_For_Target",
    ]
].rename(columns={
    "Status_Name": "To_Status",
    "Status_Group_Code": "To_Status_Group",
    "Is_Resolution_Like": "To_Is_Resolution_Like",
    "Is_Fix_Resolution": "To_Is_Fix_Resolution",
    "Is_Reopen_Destination": "To_Is_Reopen_Destination",
    "Is_Usable_For_Target": "To_Is_Usable_For_Target",
})

from_status_map = status_map_full[
    [
        "Project_ID",
        "Status_Name",
        "Status_Group_Code",
        "Is_Resolution_Like",
        "Is_Fix_Resolution",
        "Is_Reopen_Destination",
        "Is_Usable_For_Target",
    ]
].rename(columns={
    "Status_Name": "From_Status",
    "Status_Group_Code": "From_Status_Group",
    "Is_Resolution_Like": "From_Is_Resolution_Like",
    "Is_Fix_Resolution": "From_Is_Fix_Resolution",
    "Is_Reopen_Destination": "From_Is_Reopen_Destination",
    "Is_Usable_For_Target": "From_Is_Usable_For_Target",
})

In [21]:
status_changes_mapped = (
    status_changes
    .merge(
        to_status_map,
        on=["Project_ID", "To_Status"],
        how="left"
    )
    .merge(
        from_status_map,
        on=["Project_ID", "From_Status"],
        how="left"
    )
)

status_changes_mapped.shape

(271170, 17)

In [22]:
status_changes_mapped[
    [
        "To_Status_Group",
        "From_Status_Group",
        "To_Is_Resolution_Like",
        "To_Is_Fix_Resolution",
        "To_Is_Reopen_Destination",
    ]
].isna().sum()

To_Status_Group             0
From_Status_Group           0
To_Is_Resolution_Like       0
To_Is_Fix_Resolution        0
To_Is_Reopen_Destination    0
dtype: int64

In [23]:
status_changes_mapped.groupby("To_Status_Group").size().sort_values(ascending=False)

To_Status_Group
resolution         129879
validation          86814
active              41815
explicit_reopen     12662
dtype: int64

In [24]:
missing_to_status = (
    status_changes_mapped[status_changes_mapped["To_Status_Group"].isna()]
    .groupby(["Project_ID", "To_Status"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["Project_ID", "n"], ascending=[True, False])
)

missing_from_status = (
    status_changes_mapped[status_changes_mapped["From_Status_Group"].isna()]
    .groupby(["Project_ID", "From_Status"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["Project_ID", "n"], ascending=[True, False])
)

display(missing_to_status)
display(missing_from_status)

,Project_ID,To_Status,n


,Project_ID,From_Status,n


## 7. Identificación de la primera resolución

Para cada bug se identifica el primer cambio de estado que representa una resolución válida.

Se considera primera resolución al primer cambio cuyo estado destino pertenece a un grupo marcado como resolución y, además, corresponde a una resolución efectiva del bug (`Is_Fix_Resolution = 1`).

Cuando un bug tiene más de una transición de resolución, se conserva la primera según fecha de cambio y, en caso de empate, según identificador del cambio.

In [25]:
resolution_candidates = status_changes_mapped[
    (status_changes_mapped["To_Is_Resolution_Like"] == 1) &
    (status_changes_mapped["To_Is_Fix_Resolution"] == 1) &
    (status_changes_mapped["Change_Date"].notna())
].copy()

resolution_candidates.shape

(129879, 17)

De los bugs inicialmente cargados, se conservan para la construcción del target aquellos que presentan al menos una transición hacia un estado de resolución válida. Los cambios hacia estados de resolución pueden ocurrir más de una vez para un mismo bug; por eso primero se identifican todos los eventos candidatos y luego se conserva únicamente el primer evento de resolución de cada bug.

Los bugs sin una primera resolución válida no pueden utilizarse para este problema, ya que la pregunta de investigación requiere evaluar si el bug se reabre luego de haber sido resuelto.

In [26]:
first_resolution = (
    resolution_candidates
    .sort_values(
        [
            "Issue_ID",
            "Project_ID",
            "Change_Date",
            "Change_ID"
        ]
    )
    .drop_duplicates(
        ["Issue_ID", "Project_ID"],
        keep="first"
    )
    [[
        "Issue_ID",
        "Project_ID",
        "Change_ID",
        "Change_Date",
        "Author_ID",
        "From_Status",
        "From_Status_Group",
        "To_Status",
        "To_Status_Group"
    ]]
    .rename(columns={
        "Change_ID": "First_Resolution_Change_ID",
        "Change_Date": "First_Resolution_Date",
        "Author_ID": "Resolver_ID_At_First_Resolution",
        "From_Status": "Status_Before_First_Resolution",
        "From_Status_Group": "Status_Group_Before_First_Resolution",
        "To_Status": "First_Resolution_Status",
        "To_Status_Group": "First_Resolution_Group"
    })
)

first_resolution.shape

(67806, 9)

In [27]:
first_resolution_by_project = (
    first_resolution
    .groupby("Project_ID")
    .size()
    .reset_index(name="n_bugs_with_first_resolution")
    .sort_values("Project_ID")
)

first_resolution_by_project

,Project_ID,n_bugs_with_first_resolution
0,7,601
1,8,5055
2,9,381
3,12,14586
4,13,3169
5,34,33581
6,36,5360
7,42,5073


## 8. Identificación de reapertura posterior a la primera resolución

Una vez identificada la primera resolución de cada bug, se busca si existe algún cambio de estado posterior hacia un estado activo o explícitamente reabierto.

Se considera que un bug fue reabierto si, luego de su primera resolución, aparece una transición cuyo estado destino pertenece a alguno de estos grupos funcionales:

- `active`
- `explicit_reopen`

Si existen varias transiciones posteriores de este tipo, se conserva la primera.

In [28]:
status_after_resolution = status_changes_mapped.merge(
    first_resolution[
        [
            "Issue_ID",
            "Project_ID",
            "First_Resolution_Change_ID",
            "First_Resolution_Date"
        ]
    ],
    on=["Issue_ID", "Project_ID"],
    how="inner"
)

status_after_resolution.shape

(266580, 19)

In [29]:
reopen_candidates = status_after_resolution[
    (
        status_after_resolution["Change_Date"] > status_after_resolution["First_Resolution_Date"]
    )
    |
    (
        (status_after_resolution["Change_Date"] == status_after_resolution["First_Resolution_Date"])
        &
        (status_after_resolution["Change_ID"] > status_after_resolution["First_Resolution_Change_ID"])
    )
].copy()

reopen_candidates = reopen_candidates[
    reopen_candidates["To_Status_Group"].isin(["active", "explicit_reopen"])
].copy()

reopen_candidates.shape

(12246, 19)

In [30]:
first_reopen = (
    reopen_candidates
    .sort_values(["Issue_ID", "Project_ID", "Change_Date", "Change_ID"])
    .drop_duplicates(["Issue_ID", "Project_ID"], keep="first")
    [[
        "Issue_ID",
        "Project_ID",
        "Change_ID",
        "Change_Date",
        "To_Status",
        "To_Status_Group"
    ]]
    .rename(columns={
        "Change_ID": "First_Reopen_Change_ID",
        "Change_Date": "First_Reopen_Date",
        "To_Status": "First_Reopen_Status",
        "To_Status_Group": "First_Reopen_Group"
    })
)

first_reopen.shape

(7635, 6)

In [31]:
first_reopen_by_project = (
    first_reopen
    .groupby("Project_ID")
    .size()
    .reset_index(name="n_reopened_bugs")
    .sort_values("Project_ID")
)

first_reopen_by_project

,Project_ID,n_reopened_bugs
0,7,138
1,8,676
2,9,44
3,12,2984
4,13,893
5,34,1485
6,36,592
7,42,823


## 9. Construcción de la variable objetivo

Se construye la tabla base del problema predictivo. Cada fila representa un bug con primera resolución válida.

La variable `Reopened_Target` toma valor:

- `1`: el bug tuvo una reapertura posterior a la primera resolución.
- `0`: el bug no tuvo reapertura posterior registrada.

In [32]:
bug_target = first_resolution.merge(
    first_reopen,
    on=["Issue_ID", "Project_ID"],
    how="left"
)

bug_target["Reopened_Target"] = bug_target["First_Reopen_Change_ID"].notna().astype(int)

bug_target = bug_target[
    [
        "Issue_ID",
        "Project_ID",
        "First_Resolution_Change_ID",
        "First_Resolution_Date",
        "Resolver_ID_At_First_Resolution",
        "Status_Before_First_Resolution",
        "Status_Group_Before_First_Resolution",
        "First_Resolution_Status",
        "First_Resolution_Group",
        "Reopened_Target",
        "First_Reopen_Change_ID",
        "First_Reopen_Date",
        "First_Reopen_Status",
        "First_Reopen_Group",
    ]
]

bug_target.shape

(67806, 14)

In [33]:
same_timestamp_reopens = bug_target[
    bug_target["Reopened_Target"].eq(1)
    &
    bug_target["First_Reopen_Date"].eq(
        bug_target["First_Resolution_Date"]
    )
].copy()


same_timestamp_reopen_summary = pd.DataFrame({
    "metric": [
        "total_reabiertos",
        "reaperturas_mismo_timestamp",
        "reaperturas_timestamp_posterior"
    ],
    "value": [
        int(
            bug_target["Reopened_Target"].sum()
        ),
        len(same_timestamp_reopens),
        int(
            bug_target["Reopened_Target"].sum()
            - len(same_timestamp_reopens)
        )
    ]
})


same_timestamp_reopen_summary

,metric,value
0,total_reabiertos,7635
1,reaperturas_mismo_timestamp,195
2,reaperturas_timestamp_posterior,7440


In [34]:
assert (
    same_timestamp_reopens["First_Reopen_Change_ID"]
    >
    same_timestamp_reopens["First_Resolution_Change_ID"]
).all()

In [35]:
target_summary = pd.DataFrame({
    "metric": [
        "bugs_con_primera_resolucion",
        "bugs_reabiertos",
        "tasa_reapertura_pct"
    ],
    "value": [
        len(bug_target),
        bug_target["Reopened_Target"].sum(),
        round(100 * bug_target["Reopened_Target"].mean(), 2)
    ]
})

target_summary

,metric,value
0,bugs_con_primera_resolucion,67806.00
1,bugs_reabiertos,7635.00
2,tasa_reapertura_pct,11.26


In [36]:
target_by_project = (
    bug_target
    .groupby("Project_ID")
    .agg(
        bugs_con_primera_resolucion=("Issue_ID", "count"),
        bugs_reabiertos=("Reopened_Target", "sum")
    )
    .reset_index()
)

target_by_project["tasa_reapertura_pct"] = (
    100 * target_by_project["bugs_reabiertos"] / target_by_project["bugs_con_primera_resolucion"]
).round(2)

target_by_project.sort_values("tasa_reapertura_pct", ascending=False)

,Project_ID,bugs_con_primera_resolucion,bugs_reabiertos,tasa_reapertura_pct
4,13,3169,893,28.18
0,7,601,138,22.96
3,12,14586,2984,20.46
7,42,5073,823,16.22
1,8,5055,676,13.37
2,9,381,44,11.55
6,36,5360,592,11.04
5,34,33581,1485,4.42


In [37]:
first_reopen_status_distribution = (
    bug_target[bug_target["Reopened_Target"] == 1]
    .groupby(["Project_ID", "First_Reopen_Status", "First_Reopen_Group"])
    .size()
    .reset_index(name="n")
    .sort_values(["Project_ID", "n"], ascending=[True, False])
)

first_reopen_status_distribution

,Project_ID,First_Reopen_Status,First_Reopen_Group,n
0,7,Reopened,explicit_reopen,138
1,8,Reopened,explicit_reopen,676
2,9,Reopened,explicit_reopen,44
5,12,Reopened,explicit_reopen,2829
4,12,Open,active,151
3,12,In Progress,active,4
6,13,Reopened,explicit_reopen,893
8,34,Reopened,explicit_reopen,1278
7,34,Problem during testing,active,207
11,36,Reopened,explicit_reopen,565


## 10. Construcción del dataset analítico base

A partir de la tabla de target se incorporan atributos generales del bug disponibles en la tabla `issue`.

En esta etapa se evita utilizar campos que puedan representar información posterior al momento de predicción, como el estado final del issue, la resolución final o la última fecha de actualización.

In [38]:
# En la base analítica se incorporan aquí solo atributos inmutables.
# Los campos mutables se reconstruyen más adelante al instante de la primera resolución.
issue_base = issues[
    [
        "Issue_ID",
        "Project_ID",
        "Issue_Key",
        "Creation_Date"
    ]
].copy()

dataset = (
    bug_target
    .merge(
        issue_base,
        on=["Issue_ID", "Project_ID"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        project_observation_dates,
        on="Project_ID",
        how="left",
        validate="many_to_one",
    )
)

if dataset["Project_Observation_End_Date"].isna().any():
    missing_projects = sorted(
        dataset.loc[
            dataset["Project_Observation_End_Date"].isna(),
            "Project_ID",
        ].dropna().unique().tolist()
    )
    raise ValueError(
        "Quedaron bugs sin fecha final de observación. Proyectos: "
        f"{missing_projects}"
    )

invalid_windows = (
    dataset["Project_Observation_End_Date"]
    < dataset["First_Resolution_Date"]
)

if invalid_windows.any():
    raise ValueError(
        "Hay primeras resoluciones posteriores al fin observable "
        f"del proyecto. Cantidad: {int(invalid_windows.sum())}"
    )

dataset.shape


(67806, 17)

In [39]:
dataset.isna().sum().sort_values(ascending=False)

First_Reopen_Group                      60171
First_Reopen_Date                       60171
First_Reopen_Status                     60171
First_Reopen_Change_ID                  60171
Resolver_ID_At_First_Resolution            54
Issue_ID                                    0
Project_ID                                  0
First_Resolution_Change_ID                  0
First_Resolution_Date                       0
First_Resolution_Group                      0
First_Resolution_Status                     0
Status_Group_Before_First_Resolution        0
Status_Before_First_Resolution              0
Reopened_Target                             0
Issue_Key                                   0
Creation_Date                               0
Project_Observation_End_Date                0
dtype: int64

### Variables temporales básicas

Se construyen variables asociadas al momento de creación del bug y al tiempo transcurrido hasta la primera resolución.

Estas variables son válidas para el problema predictivo porque se calculan usando información disponible hasta el momento de la primera resolución.

In [40]:
dataset["resolution_time_hours"] = (
    dataset["First_Resolution_Date"]
    - dataset["Creation_Date"]
).dt.total_seconds() / 3600

dataset["resolution_time_days"] = (
    dataset["resolution_time_hours"] / 24
)

# Variables calendario de creación.
dataset["creation_year"] = dataset["Creation_Date"].dt.year
dataset["creation_month"] = dataset["Creation_Date"].dt.month
dataset["creation_weekday"] = dataset["Creation_Date"].dt.weekday

# Variables calendario del momento de la primera resolución.
dataset["resolution_year"] = (
    dataset["First_Resolution_Date"].dt.year
)
dataset["resolution_month"] = (
    dataset["First_Resolution_Date"].dt.month
)
dataset["resolution_quarter"] = (
    dataset["First_Resolution_Date"].dt.quarter
)
dataset["resolution_weekday"] = (
    dataset["First_Resolution_Date"].dt.weekday
)

# Codificación cíclica del mes. Enero y diciembre quedan próximos.
dataset["resolution_month_sin"] = np.sin(
    2
    * np.pi
    * dataset["resolution_month"]
    / 12
)
dataset["resolution_month_cos"] = np.cos(
    2
    * np.pi
    * dataset["resolution_month"]
    / 12
)

# Antigüedad del proyecto al resolver el bug.
project_first_resolution_date = (
    dataset
    .groupby("Project_ID")["First_Resolution_Date"]
    .transform("min")
)

dataset["days_since_project_start"] = (
    dataset["First_Resolution_Date"]
    - project_first_resolution_date
).dt.total_seconds() / 86400

dataset["project_age_years"] = (
    dataset["days_since_project_start"] / 365.25
)

basic_temporal_columns = [
    "resolution_time_hours",
    "resolution_time_days",
    "creation_year",
    "creation_month",
    "creation_weekday",
    "resolution_year",
    "resolution_month",
    "resolution_quarter",
    "resolution_weekday",
    "resolution_month_sin",
    "resolution_month_cos",
    "days_since_project_start",
    "project_age_years",
]

dataset[basic_temporal_columns].describe().T


,count,mean,std,min,25%,50%,75%,max
resolution_time_hours,67806.0,7049.030158,14109.743117,0.0,32.053611,3.849194e+02,5805.245972,118934.805556
resolution_time_days,67806.0,293.709590,587.905963,0.0,1.335567,1.603831e+01,241.885249,4955.616898
creation_year,67806.0,2012.262543,3.409453,2003.0,2011.000000,2.012000e+03,2015.000000,2020.000000
creation_month,67806.0,6.493260,3.314786,1.0,4.000000,6.000000e+00,9.000000,12.000000
creation_weekday,67806.0,2.443663,1.628548,0.0,1.000000,2.000000e+00,4.000000,6.000000
resolution_year,67806.0,2013.066852,3.314274,2004.0,2011.000000,2.013000e+03,2016.000000,2020.000000
resolution_month,67806.0,6.505619,3.350871,1.0,4.000000,6.000000e+00,9.000000,12.000000
resolution_quarter,67806.0,2.509925,1.089984,1.0,2.000000,2.000000e+00,3.000000,4.000000
resolution_weekday,67806.0,2.309810,1.603113,0.0,1.000000,2.000000e+00,3.000000,6.000000
resolution_month_sin,67806.0,0.010032,0.703868,-1.0,-0.500000,1.224647e-16,0.866025,1.000000


## 11. Historial de cambios hasta la primera resolución

Se incorporan variables derivadas del historial de cambios registrado en `change_log`.

Para cada bug se consideran únicamente los cambios ocurridos hasta el momento de su primera resolución. Esto permite construir variables disponibles al momento de predicción y evita utilizar información posterior a la resolución.

In [41]:
# Carga de change_log por bloques para evitar agotar la memoria.
# No se conserva una copia completa de todos los cambios.

import gc

KEY_COLS = ["Issue_ID", "Project_ID"]
CHANGE_CHUNK_SIZE = 150_000

resolution_cutoff = first_resolution[
    [
        "Issue_ID",
        "Project_ID",
        "First_Resolution_Date",
        "First_Resolution_Change_ID",
    ]
].copy()

# Solo estos campos requieren conservar From/To para reconstruir
# el valor vigente en la primera resolución.
RECONSTRUCTION_FIELD_ALIASES = {
    "priority",
    "creator",
    "reporter",
    "assignee",
    "story points",
    "story point",
    "summary",
    "title",
    "description",
    "component",
    "components",
    "version",
    "affected version",
    "affected versions",
    "fix version",
    "fix versions",
    "link",
    "links",
}


def classify_change_fields_vectorized(field_series):
    normalized = (
        field_series
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    groups = np.full(len(normalized), "other", dtype=object)

    missing_mask = normalized.isna().to_numpy()
    groups[missing_mask] = "missing"

    exact_groups = {
        "status": "status",
        "priority": "priority",
        "resolution": "resolution",
        "assignee": "people",
        "reporter": "people",
        "creator": "people",
        "description": "description",
        "environment": "description",
        "summary": "summary",
        "title": "summary",
        "component": "component",
        "components": "component",
        "version": "affected_version",
        "affected version": "affected_version",
        "affected versions": "affected_version",
        "fix version": "fix_version",
        "fix versions": "fix_version",
        "link": "link",
        "links": "link",
    }

    for field_name, group_name in exact_groups.items():
        groups[normalized.eq(field_name).fillna(False).to_numpy()] = group_name

    story_point_mask = (
        normalized.str.contains("story", na=False)
        & normalized.str.contains("point", na=False)
    ).to_numpy()
    groups[story_point_mask] = "story_point"

    return normalized, groups


change_count_parts = []
change_field_count_parts = []
change_author_parts = []
reconstruction_parts = []

source_change_rows_read = 0
changes_before_resolution_rows = 0

change_sql = f"""
SELECT
    cl.ID AS Change_ID,
    cl.Issue_ID,
    i.Project_ID,
    cl.Creation_Date AS Change_Date,
    cl.Field,
    cl.From_Value,
    cl.To_Value,
    cl.From_String,
    cl.To_String,
    cl.Author_ID
FROM change_log cl
JOIN `issue` i
    ON i.ID = cl.Issue_ID
WHERE i.Type = 'Bug'
  AND i.Project_ID IN ({project_ids_sql})
ORDER BY cl.ID
"""

with engine.connect().execution_options(
    stream_results=True,
    max_row_buffer=CHANGE_CHUNK_SIZE,
) as connection:
    chunk_iterator = pd.read_sql(
        change_sql,
        connection,
        parse_dates=["Change_Date"],
        chunksize=CHANGE_CHUNK_SIZE,
    )

    for chunk_number, chunk in enumerate(chunk_iterator, start=1):
        source_change_rows_read += len(chunk)

        # El inner join elimina cambios de bugs que no pertenecen a la
        # cohorte con primera resolución reconstruida.
        scoped = chunk.merge(
            resolution_cutoff,
            on=KEY_COLS,
            how="inner",
            validate="many_to_one",
        )

        if scoped.empty:
            del chunk, scoped
            gc.collect()
            continue

        normalized_field, field_groups = (
            classify_change_fields_vectorized(scoped["Field"])
        )
        scoped["_Field_Normalized"] = normalized_field
        scoped["_Change_Field_Group"] = field_groups

        before_mask = (
            (scoped["Change_Date"] < scoped["First_Resolution_Date"])
            |
            (
                scoped["Change_Date"].eq(scoped["First_Resolution_Date"])
                & scoped["Change_ID"].le(
                    scoped["First_Resolution_Change_ID"]
                )
            )
        )

        before = scoped.loc[
            before_mask,
            [
                "Change_ID",
                "Issue_ID",
                "Project_ID",
                "Change_Date",
                "Author_ID",
                "_Change_Field_Group",
            ],
        ].copy()

        changes_before_resolution_rows += len(before)

        if not before.empty:
            change_count_parts.append(
                before.groupby(KEY_COLS, sort=False)
                .size()
                .rename("n_changes_until_resolution")
                .reset_index()
            )

            change_field_count_parts.append(
                before.groupby(
                    KEY_COLS + ["_Change_Field_Group"],
                    sort=False,
                )
                .size()
                .rename("n")
                .reset_index()
            )

            author_pairs = (
                before.loc[
                    before["Author_ID"].notna(),
                    KEY_COLS + ["Author_ID"],
                ]
                .drop_duplicates()
            )
            if not author_pairs.empty:
                change_author_parts.append(author_pairs)

        reconstruction_mask = scoped[
            "_Field_Normalized"
        ].isin(RECONSTRUCTION_FIELD_ALIASES)

        reconstruction_piece = scoped.loc[
            reconstruction_mask,
            [
                "Change_ID",
                "Issue_ID",
                "Project_ID",
                "Change_Date",
                "Field",
                "From_Value",
                "To_Value",
                "From_String",
                "To_String",
                "Author_ID",
            ],
        ].copy()

        if not reconstruction_piece.empty:
            reconstruction_parts.append(reconstruction_piece)

        if chunk_number % 10 == 0:
            print(
                f"Bloques leídos: {chunk_number} | "
                f"filas fuente: {source_change_rows_read:,} | "
                f"filas previas a resolución: "
                f"{changes_before_resolution_rows:,}"
            )

        del chunk, scoped, before, reconstruction_piece
        gc.collect()

# Conteo total de cambios anteriores a la primera resolución.
change_features_general = (
    pd.concat(change_count_parts, ignore_index=True)
    .groupby(KEY_COLS, as_index=False, sort=False)[
        "n_changes_until_resolution"
    ]
    .sum()
)

# Cantidad de autores distintos. Se deduplican también entre bloques.
if change_author_parts:
    change_author_pairs = (
        pd.concat(change_author_parts, ignore_index=True)
        .drop_duplicates(KEY_COLS + ["Author_ID"])
    )

    change_author_counts = (
        change_author_pairs
        .groupby(KEY_COLS, as_index=False, sort=False)
        .size()
        .rename(columns={"size": "n_change_authors_until_resolution"})
    )

    change_features_general = change_features_general.merge(
        change_author_counts,
        on=KEY_COLS,
        how="left",
        validate="one_to_one",
    )
else:
    change_features_general[
        "n_change_authors_until_resolution"
    ] = 0

change_features_general[
    "n_change_authors_until_resolution"
] = (
    change_features_general[
        "n_change_authors_until_resolution"
    ]
    .fillna(0)
    .astype("int64")
)

# Conteos por grupo de campo.
change_field_long = (
    pd.concat(change_field_count_parts, ignore_index=True)
    .groupby(
        KEY_COLS + ["_Change_Field_Group"],
        as_index=False,
        sort=False,
    )["n"]
    .sum()
)

change_field_features = (
    change_field_long
    .pivot_table(
        index=KEY_COLS,
        columns="_Change_Field_Group",
        values="n",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

change_field_features.columns.name = None
change_field_features = change_field_features.rename(
    columns={
        col: f"n_change_{col}_until_resolution"
        for col in change_field_features.columns
        if col not in KEY_COLS
    }
)

# Única tabla de cambios detallados que se conserva: solamente los
# campos necesarios para reconstrucciones escalares y estructurales.
if reconstruction_parts:
    reconstruction_changes = pd.concat(
        reconstruction_parts,
        ignore_index=True,
    )
else:
    reconstruction_changes = pd.DataFrame(
        columns=[
            "Change_ID",
            "Issue_ID",
            "Project_ID",
            "Change_Date",
            "Field",
            "From_Value",
            "To_Value",
            "From_String",
            "To_String",
            "Author_ID",
        ]
    )

reconstruction_changes = reconstruction_changes.sort_values(
    KEY_COLS + ["Change_Date", "Change_ID"],
    kind="mergesort",
).reset_index(drop=True)

# Liberación de acumuladores temporales.
del (
    change_count_parts,
    change_field_count_parts,
    change_author_parts,
    reconstruction_parts,
    change_field_long,
)
if "change_author_pairs" in globals():
    del change_author_pairs
if "change_author_counts" in globals():
    del change_author_counts

gc.collect()

changes_until_resolution_temporal_ok = True

print("Filas de change_log leídas:", f"{source_change_rows_read:,}")
print(
    "Cambios anteriores a la primera resolución:",
    f"{changes_before_resolution_rows:,}",
)
print(
    "Cambios detallados conservados para reconstrucción:",
    f"{len(reconstruction_changes):,}",
)
print("Features generales:", change_features_general.shape)
print("Features por grupo de campo:", change_field_features.shape)


Bloques leídos: 10 | filas fuente: 1,500,000 | filas previas a resolución: 963,636
Filas de change_log leídas: 1,526,172
Cambios anteriores a la primera resolución: 979,802
Cambios detallados conservados para reconstrucción: 392,421
Features generales: (67806, 4)
Features por grupo de campo: (67806, 14)


In [42]:
# La tabla completa changes_until_resolution ya no se materializa.
# Sus agregados fueron calculados por bloques en la celda anterior.
display(change_features_general.head())
display(change_field_features.head())
reconstruction_changes.shape


,Issue_ID,Project_ID,n_changes_until_resolution,n_change_authors_until_resolution
0,27390,7,16,3
1,27392,7,6,2
2,27394,7,19,3
3,27407,7,9,2
4,27408,7,9,3


,Issue_ID,Project_ID,n_change_affected_version_until_resolution,n_change_component_until_resolution,n_change_description_until_resolution,n_change_fix_version_until_resolution,n_change_link_until_resolution,n_change_other_until_resolution,n_change_people_until_resolution,n_change_priority_until_resolution,n_change_resolution_until_resolution,n_change_status_until_resolution,n_change_story_point_until_resolution,n_change_summary_until_resolution
0,27390,7,2,0,2,1,1,4,1,0,0,4,0,1
1,27392,7,0,0,1,1,0,0,0,0,0,4,0,0
2,27394,7,0,0,7,1,0,2,2,0,0,4,0,3
3,27407,7,0,0,1,1,1,1,0,1,0,4,0,0
4,27408,7,0,0,2,1,0,1,1,0,0,4,0,0


(392421, 10)

In [43]:
NULL_TEXT_VALUES = {
    "",
    "null",
    "none",
    "nan",
    "<na>"
}


def clean_nullable_text(series):
    cleaned = (
        series
        .astype("string")
        .str.strip()
    )

    return cleaned.mask(
        cleaned.str.casefold().isin(NULL_TEXT_VALUES)
    )


def coalesce_history_columns(df, columns):
    result = pd.Series(pd.NA, index=df.index, dtype="string")

    for col in columns:
        if col not in df.columns:
            continue
        result = result.combine_first(clean_nullable_text(df[col]))

    return result


def is_at_or_before_first_resolution(df):
    return (
        (df["Change_Date"] < df["First_Resolution_Date"])
        |
        (
            (df["Change_Date"] == df["First_Resolution_Date"])
            &
            (df["Change_ID"] <= df["First_Resolution_Change_ID"])
        )
    )


def is_after_first_resolution(df):
    return (
        (df["Change_Date"] > df["First_Resolution_Date"])
        |
        (
            (df["Change_Date"] == df["First_Resolution_Date"])
            &
            (df["Change_ID"] > df["First_Resolution_Change_ID"])
        )
    )


def nullable_values_equal(left, right):
    left_clean = clean_nullable_text(left)
    right_clean = clean_nullable_text(right)

    return (
        left_clean.eq(right_clean)
        |
        (left_clean.isna() & right_clean.isna())
    ).fillna(False)


def reconstruct_scalar_at_resolution(
    field_names,
    current_column,
    output_column,
    from_columns,
    to_columns
):
    """
    Reconstruye el valor vigente al momento de la primera resolución.

    Orden de prioridad:
      1. To del último cambio ocurrido antes o en la primera resolución.
      2. From del primer cambio posterior a la primera resolución.
      3. Valor de issue únicamente si el campo nunca aparece en change_log.

    La existencia del cambio se controla por separado del valor, porque un
    valor nulo puede ser un estado real del campo (por ejemplo, sin assignee).
    """

    if isinstance(field_names, str):
        field_names = [field_names]

    aliases = {
        str(name).strip().casefold()
        for name in field_names
    }

    current_values = (
        bug_target[KEY_COLS]
        .merge(
            issues[KEY_COLS + [current_column]],
            on=KEY_COLS,
            how="left",
            validate="one_to_one"
        )
        .rename(columns={current_column: "_current_value"})
    )

    normalized_field = (
        reconstruction_changes["Field"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    field_changes = reconstruction_changes[
        normalized_field.isin(aliases)
    ].copy()

    field_changes["_value_before_change"] = coalesce_history_columns(
        field_changes,
        from_columns
    )

    field_changes["_value_after_change"] = coalesce_history_columns(
        field_changes,
        to_columns
    )

    field_changes = field_changes.merge(
        resolution_cutoff,
        on=KEY_COLS,
        how="inner",
        validate="many_to_one"
    )

    changes_before = field_changes[
        is_at_or_before_first_resolution(field_changes)
    ].copy()

    last_change_before = (
        changes_before
        .sort_values(KEY_COLS + ["Change_Date", "Change_ID"])
        .drop_duplicates(KEY_COLS, keep="last")
        [
            KEY_COLS
            + [
                "Change_ID",
                "Change_Date",
                "_value_after_change"
            ]
        ]
        .rename(
            columns={
                "Change_ID": "_last_change_before_id",
                "Change_Date": "_last_change_before_date",
                "_value_after_change": "_value_from_last_change_before"
            }
        )
    )
    last_change_before["_has_change_before"] = True

    changes_after = field_changes[
        is_after_first_resolution(field_changes)
    ].copy()

    first_change_after = (
        changes_after
        .sort_values(KEY_COLS + ["Change_Date", "Change_ID"])
        .drop_duplicates(KEY_COLS, keep="first")
        [
            KEY_COLS
            + [
                "Change_ID",
                "Change_Date",
                "_value_before_change"
            ]
        ]
        .rename(
            columns={
                "Change_ID": "_first_change_after_id",
                "Change_Date": "_first_change_after_date",
                "_value_before_change": "_value_from_first_change_after"
            }
        )
    )
    first_change_after["_has_change_after"] = True

    result = (
        current_values
        .merge(
            last_change_before,
            on=KEY_COLS,
            how="left",
            validate="one_to_one"
        )
        .merge(
            first_change_after,
            on=KEY_COLS,
            how="left",
            validate="one_to_one"
        )
    )

    has_before = result["_has_change_before"].eq(True)
    has_after = result["_has_change_after"].eq(True)

    # El valor puede ser nulo de forma legítima. Por eso no se usa combine_first
    # para decidir la fuente: se usa la existencia de una fila de cambio.
    result[output_column] = clean_nullable_text(result["_current_value"])

    use_after = (~has_before) & has_after
    result.loc[use_after, output_column] = result.loc[
        use_after,
        "_value_from_first_change_after"
    ]

    result.loc[has_before, output_column] = result.loc[
        has_before,
        "_value_from_last_change_before"
    ]

    result["_reconstruction_source"] = "issue_without_change_log"
    result.loc[use_after, "_reconstruction_source"] = "first_change_after_from"
    result.loc[has_before, "_reconstruction_source"] = "last_change_before_to"

    both_sides = has_before & has_after

    consistent_around_cutoff = pd.Series(
        True,
        index=result.index,
        dtype="boolean"
    )

    consistent_around_cutoff.loc[both_sides] = nullable_values_equal(
        result.loc[both_sides, "_value_from_last_change_before"],
        result.loc[both_sides, "_value_from_first_change_after"]
    ).values

    inconsistent_mask = (
        both_sides
        &
        ~consistent_around_cutoff.fillna(False)
    )

    inconsistencies = result.loc[
        inconsistent_mask,
        KEY_COLS
        + [
            "_last_change_before_id",
            "_last_change_before_date",
            "_value_from_last_change_before",
            "_first_change_after_id",
            "_first_change_after_date",
            "_value_from_first_change_after"
        ]
    ].copy()

    inconsistencies.insert(0, "Field_Output", output_column)

    audit = {
        "Field_Output": output_column,
        "n_issues": len(result),
        "source_last_change_before_to": int(has_before.sum()),
        "source_first_change_after_from": int(use_after.sum()),
        "source_issue_without_change_log": int(
            ((~has_before) & (~has_after)).sum()
        ),
        "history_on_both_sides": int(both_sides.sum()),
        "inconsistencies_around_resolution": int(
            inconsistent_mask.sum()
        ),
        "selected_null_values": int(result[output_column].isna().sum())
    }

    features = result[KEY_COLS + [output_column]].copy()

    return features, audit, inconsistencies


In [44]:
SCALAR_FIELD_CONFIGS = [
    {
        "field_names": "priority",
        "current_column": "Priority",
        "output_column": "Priority_At_First_Resolution",
        "from_columns": ["From_String", "From_Value"],
        "to_columns": ["To_String", "To_Value"]
    },
    {
        "field_names": "creator",
        "current_column": "Creator_ID",
        "output_column": "Creator_ID_At_First_Resolution",
        "from_columns": ["From_Value", "From_String"],
        "to_columns": ["To_Value", "To_String"]
    },
    {
        "field_names": "reporter",
        "current_column": "Reporter_ID",
        "output_column": "Reporter_ID_At_First_Resolution",
        "from_columns": ["From_Value", "From_String"],
        "to_columns": ["To_Value", "To_String"]
    },
    {
        "field_names": "assignee",
        "current_column": "Assignee_ID",
        "output_column": "Assignee_ID_At_First_Resolution",
        "from_columns": ["From_Value", "From_String"],
        "to_columns": ["To_Value", "To_String"]
    },
    {
        "field_names": ["Story Points", "Story Point"],
        "current_column": "Story_Point",
        "output_column": "Story_Point_At_First_Resolution",
        "from_columns": ["From_String", "From_Value"],
        "to_columns": ["To_String", "To_Value"]
    },
    {
        "field_names": ["summary", "title"],
        "current_column": "Title",
        "output_column": "Title_At_First_Resolution",
        "from_columns": ["From_String", "From_Value"],
        "to_columns": ["To_String", "To_Value"]
    },
    {
        "field_names": "description",
        "current_column": "Description",
        "output_column": "Description_At_First_Resolution",
        "from_columns": ["From_String", "From_Value"],
        "to_columns": ["To_String", "To_Value"]
    }
]


scalar_feature_tables = []
scalar_audit_rows = []
scalar_inconsistency_tables = []

for config in SCALAR_FIELD_CONFIGS:
    feature_table, audit_row, inconsistency_table = (
        reconstruct_scalar_at_resolution(**config)
    )

    scalar_feature_tables.append(feature_table)
    scalar_audit_rows.append(audit_row)

    if not inconsistency_table.empty:
        scalar_inconsistency_tables.append(inconsistency_table)


scalar_reconstruction_audit = pd.DataFrame(scalar_audit_rows)

if scalar_inconsistency_tables:
    scalar_history_inconsistencies = pd.concat(
        scalar_inconsistency_tables,
        ignore_index=True
    )
else:
    scalar_history_inconsistencies = pd.DataFrame(
        columns=[
            "Field_Output",
            "Issue_ID",
            "Project_ID",
            "_last_change_before_id",
            "_last_change_before_date",
            "_value_from_last_change_before",
            "_first_change_after_id",
            "_first_change_after_date",
            "_value_from_first_change_after"
        ]
    )

display(scalar_reconstruction_audit)
display(scalar_history_inconsistencies.head(50))

if not scalar_history_inconsistencies.empty:
    print(
        "ADVERTENCIA: hay discontinuidades entre el último To anterior "
        "y el primer From posterior a la resolución. Revisar esos casos."
    )


for feature_table in scalar_feature_tables:
    dataset = dataset.merge(
        feature_table,
        on=KEY_COLS,
        how="left",
        validate="one_to_one"
    )

dataset.shape


,Field_Output,n_issues,source_last_change_before_to,source_first_change_after_from,source_issue_without_change_log,history_on_both_sides,inconsistencies_around_resolution,selected_null_values
0,Priority_At_First_Resolution,67806,21391,3988,42427,441,14,1730
1,Creator_ID_At_First_Resolution,67806,0,0,67806,0,0,175
2,Reporter_ID_At_First_Resolution,67806,551,125,67130,4,0,0
3,Assignee_ID_At_First_Resolution,67806,39380,2880,25546,4462,39,10575
4,Story_Point_At_First_Resolution,67806,5289,1734,60783,84,0,60447
5,Title_At_First_Resolution,67806,9897,1156,56753,234,6,1
6,Description_At_First_Resolution,67806,13243,579,53984,211,54,707


,Field_Output,Issue_ID,Project_ID,_last_change_before_id,_last_change_before_date,_value_from_last_change_before,_first_change_after_id,_first_change_after_date,_value_from_first_change_after
0,Priority_At_First_Resolution,36521,8,528190.0,2011-06-08 03:06:11,Major,528198.0,2011-08-22 11:49:38,Medium
1,Priority_At_First_Resolution,420631,12,8519373.0,2011-06-05 11:27:26,Major,8519405.0,2012-03-19 21:36:02,Medium
2,Priority_At_First_Resolution,420732,12,8521103.0,2011-05-20 15:03:03,Critical,8521132.0,2012-01-27 17:25:28,High
3,Priority_At_First_Resolution,420733,12,8521144.0,2011-05-20 15:04:09,Critical,8521171.0,2012-02-14 16:37:48,High
4,Priority_At_First_Resolution,421507,12,8534019.0,2011-04-15 03:39:30,Major,8534038.0,2011-07-25 15:39:22,Medium
5,Priority_At_First_Resolution,421635,12,8536563.0,2011-06-15 12:43:38,Minor,8536584.0,2012-06-08 15:49:54,Low
6,Priority_At_First_Resolution,421641,12,8536672.0,2011-04-15 03:36:00,Major,8536707.0,2012-03-11 22:21:48,Medium
7,Priority_At_First_Resolution,422438,12,8550882.0,2011-04-15 03:14:51,Minor,8550895.0,2012-07-12 21:51:14,Low
8,Priority_At_First_Resolution,422781,12,8556793.0,2011-04-15 03:05:08,Major,8556810.0,2012-03-27 16:07:54,Medium
9,Priority_At_First_Resolution,422854,12,8558079.0,2011-04-15 03:03:16,Major,8558131.0,2012-03-11 22:20:14,Medium


ADVERTENCIA: hay discontinuidades entre el último To anterior y el primer From posterior a la resolución. Revisar esos casos.


(67806, 37)

In [45]:
# La reconstrucción y el merge de campos escalares se realizan en el bloque anterior.


In [46]:
def nullable_numeric(series):
    cleaned = clean_nullable_text(series)
    return pd.to_numeric(cleaned, errors="coerce")


USER_ID_COLUMNS = [
    "Creator_ID_At_First_Resolution",
    "Reporter_ID_At_First_Resolution",
    "Assignee_ID_At_First_Resolution",
    "Resolver_ID_At_First_Resolution"
]


for col in USER_ID_COLUMNS:
    dataset[col] = (
        nullable_numeric(dataset[col])
        .astype("Int64")
    )


dataset["Story_Point_At_First_Resolution"] = nullable_numeric(
    dataset["Story_Point_At_First_Resolution"]
)


for col in [
    "Priority_At_First_Resolution",
    "Title_At_First_Resolution",
    "Description_At_First_Resolution"
]:
    dataset[col] = clean_nullable_text(dataset[col])


dataset[
    [
        "Priority_At_First_Resolution",
        "Creator_ID_At_First_Resolution",
        "Reporter_ID_At_First_Resolution",
        "Assignee_ID_At_First_Resolution",
        "Resolver_ID_At_First_Resolution",
        "Story_Point_At_First_Resolution",
        "Title_At_First_Resolution",
        "Description_At_First_Resolution"
    ]
].isna().sum()

Priority_At_First_Resolution        1730
Creator_ID_At_First_Resolution       175
Reporter_ID_At_First_Resolution      187
Assignee_ID_At_First_Resolution    12314
Resolver_ID_At_First_Resolution       54
Story_Point_At_First_Resolution    60447
Title_At_First_Resolution              1
Description_At_First_Resolution      707
dtype: int64

### Variables de completitud y contenido al momento de la primera resolución

Las longitudes y los indicadores de contenido se calculan sobre el título y la descripción
reconstruidos al momento de la primera resolución. No se utilizan las versiones finales
almacenadas en `issue`.

In [47]:
title_at_cutoff = (
    dataset["Title_At_First_Resolution"]
    .fillna("")
    .astype(str)
)

description_at_cutoff = (
    dataset["Description_At_First_Resolution"]
    .fillna("")
    .astype(str)
)

dataset["title_length"] = title_at_cutoff.str.strip().str.len()
dataset["has_title"] = (dataset["title_length"] > 0).astype(int)

dataset["description_text_length"] = (
    description_at_cutoff
    .str.strip()
    .str.len()
)
dataset["has_description_text"] = (
    dataset["description_text_length"] > 0
).astype(int)


CODE_BLOCK_PATTERN = re.compile(
    r"```.*?```"
    r"|\{code(?::[^}]*)?\}.*?\{code\}"
    r"|\{noformat\}.*?\{noformat\}",
    flags=re.IGNORECASE | re.DOTALL
)


def extracted_code_length(text_value):
    if not text_value:
        return 0
    return sum(
        len(match.group(0))
        for match in CODE_BLOCK_PATTERN.finditer(text_value)
    )


dataset["description_code_length"] = (
    description_at_cutoff
    .map(extracted_code_length)
    .astype("int64")
)

dataset["has_description_code"] = (
    dataset["description_code_length"] > 0
).astype(int)


PULL_REQUEST_PATTERN = (
    r"https?://[^\s\])}>]+/(?:pull|pulls|merge_requests)/\d+"
    r"|\bpull request\b"
    r"|\bmerge request\b"
)

dataset["has_pull_request"] = (
    description_at_cutoff
    .str.contains(
        PULL_REQUEST_PATTERN,
        case=False,
        regex=True,
        na=False
    )
    .astype(int)
)

dataset[
    [
        "has_title",
        "title_length",
        "has_description_text",
        "description_text_length",
        "has_description_code",
        "description_code_length",
        "has_pull_request"
    ]
].describe()

,has_title,title_length,has_description_text,description_text_length,has_description_code,description_code_length,has_pull_request
count,67806.000000,67806.000000,67806.000000,6.780600e+04,67806.000000,67806.000000,67806.00000
mean,0.999985,62.867858,0.989573,9.866585e+02,0.167242,315.947866,0.00295
std,0.003840,24.447014,0.101579,7.832354e+03,0.373194,4297.729566,0.05423
min,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.00000
25%,1.000000,46.000000,1.000000,2.440000e+02,0.000000,0.000000,0.00000
50%,1.000000,60.000000,1.000000,4.660000e+02,0.000000,0.000000,0.00000
75%,1.000000,76.000000,1.000000,8.810000e+02,0.000000,0.000000,0.00000
max,1.000000,271.000000,1.000000,1.391306e+06,1.000000,499930.000000,1.00000


### Variables asociadas a usuarios al momento de la primera resolución

Los indicadores utilizan los identificadores reconstruidos en la primera resolución,
no los valores finales almacenados en la tabla `issue`.

In [48]:
creator_col = "Creator_ID_At_First_Resolution"
reporter_col = "Reporter_ID_At_First_Resolution"
assignee_col = "Assignee_ID_At_First_Resolution"
resolver_col = "Resolver_ID_At_First_Resolution"


dataset["has_creator"] = (
    dataset[creator_col].notna().astype(int)
)

dataset["has_reporter"] = (
    dataset[reporter_col].notna().astype(int)
)

dataset["has_assignee"] = (
    dataset[assignee_col].notna().astype(int)
)

dataset["has_resolver"] = (
    dataset[resolver_col].notna().astype(int)
)


dataset["creator_eq_reporter"] = (
    dataset[creator_col].notna()
    &
    dataset[reporter_col].notna()
    &
    dataset[creator_col].eq(dataset[reporter_col])
).fillna(False).astype(int)


dataset["creator_eq_assignee"] = (
    dataset[creator_col].notna()
    &
    dataset[assignee_col].notna()
    &
    dataset[creator_col].eq(dataset[assignee_col])
).fillna(False).astype(int)


dataset["reporter_eq_assignee"] = (
    dataset[reporter_col].notna()
    &
    dataset[assignee_col].notna()
    &
    dataset[reporter_col].eq(dataset[assignee_col])
).fillna(False).astype(int)


dataset["resolver_eq_assignee"] = (
    dataset[resolver_col].notna()
    &
    dataset[assignee_col].notna()
    &
    dataset[resolver_col].eq(dataset[assignee_col])
).fillna(False).astype(int)


dataset["resolver_eq_reporter"] = (
    dataset[resolver_col].notna()
    &
    dataset[reporter_col].notna()
    &
    dataset[resolver_col].eq(dataset[reporter_col])
).fillna(False).astype(int)


dataset["resolver_eq_creator"] = (
    dataset[resolver_col].notna()
    &
    dataset[creator_col].notna()
    &
    dataset[resolver_col].eq(dataset[creator_col])
).fillna(False).astype(int)


USER_RELATIONSHIP_COLUMNS = [
    "creator_eq_reporter",
    "creator_eq_assignee",
    "reporter_eq_assignee",
    "resolver_eq_assignee",
    "resolver_eq_reporter",
    "resolver_eq_creator"
]


dataset[
    [
        "has_creator",
        "has_reporter",
        "has_assignee",
        "has_resolver"
    ]
    + USER_RELATIONSHIP_COLUMNS
].mean().sort_values(ascending=False)

has_resolver            0.999204
has_creator             0.997419
has_reporter            0.997242
creator_eq_reporter     0.991284
has_assignee            0.818394
resolver_eq_assignee    0.387090
creator_eq_assignee     0.180825
reporter_eq_assignee    0.180250
resolver_eq_creator     0.131301
resolver_eq_reporter    0.130932
dtype: float64

In [49]:
def summarize_user_ids(df, columns):
    rows = []

    for col in columns:
        values = df[col].dropna()
        counts = values.value_counts()

        if len(values) == 0:
            rows.append({
                "variable": col,
                "n_non_null": 0,
                "n_unique_ids": 0,
                "unique_ratio_pct": 0,
                "median_issues_per_id": 0,
                "max_issues_per_id": 0,
                "ids_with_1_issue": 0,
                "rows_from_singleton_ids_pct": 0,
                "ids_with_at_least_5_issues": 0,
                "ids_with_at_least_20_issues": 0,
                "top_10_ids_share_pct": 0
            })
            continue

        singleton_rows = counts[counts == 1].sum()

        rows.append({
            "variable": col,
            "n_non_null": len(values),
            "n_unique_ids": values.nunique(),
            "unique_ratio_pct": round(
                100 * values.nunique() / len(values),
                2
            ),
            "median_issues_per_id": float(
                counts.median()
            ),
            "max_issues_per_id": int(
                counts.max()
            ),
            "ids_with_1_issue": int(
                (counts == 1).sum()
            ),
            "rows_from_singleton_ids_pct": round(
                100 * singleton_rows / len(values),
                2
            ),
            "ids_with_at_least_5_issues": int(
                (counts >= 5).sum()
            ),
            "ids_with_at_least_20_issues": int(
                (counts >= 20).sum()
            ),
            "top_10_ids_share_pct": round(
                100 * counts.head(10).sum() / len(values),
                2
            )
        })

    return pd.DataFrame(rows)


user_id_summary = summarize_user_ids(
    dataset,
    USER_ID_COLUMNS
)

user_id_summary

,variable,n_non_null,n_unique_ids,unique_ratio_pct,median_issues_per_id,max_issues_per_id,ids_with_1_issue,rows_from_singleton_ids_pct,ids_with_at_least_5_issues,ids_with_at_least_20_issues,top_10_ids_share_pct
0,Creator_ID_At_First_Resolution,67631,11639,17.21,1.0,934,7427,10.98,1553,492,9.55
1,Reporter_ID_At_First_Resolution,67619,11592,17.14,1.0,936,7397,10.94,1556,496,9.56
2,Assignee_ID_At_First_Resolution,55492,823,1.48,5.0,3529,205,0.37,429,239,30.89
3,Resolver_ID_At_First_Resolution,67752,838,1.24,6.0,3593,241,0.36,440,266,26.15


In [50]:
def add_previous_role_count_in_project(
    df,
    id_column,
    output_column
):
    """
    Cuenta cuántos bugs con una primera resolución anterior
    tuvo la misma persona dentro del mismo proyecto.

    No utiliza Reopened_Target ni información posterior.
    """

    result = pd.Series(
        0,
        index=df.index,
        dtype="int64"
    )

    valid_mask = (
        df[id_column].notna()
        &
        df["First_Resolution_Date"].notna()
        &
        df["First_Resolution_Change_ID"].notna()
    )

    ordered = df.loc[
        valid_mask,
        [
            "Issue_ID",
            "Project_ID",
            "First_Resolution_Date",
            "First_Resolution_Change_ID",
            id_column
        ]
    ].copy()

    ordered["_original_index"] = ordered.index

    ordered = ordered.sort_values(
        [
            "Project_ID",
            id_column,
            "First_Resolution_Date",
            "First_Resolution_Change_ID",
            "Issue_ID"
        ],
        kind="mergesort"
    )

    ordered[output_column] = (
        ordered
        .groupby(
            ["Project_ID", id_column],
            sort=False
        )
        .cumcount()
        .astype("int64")
    )

    result.loc[
        ordered["_original_index"]
    ] = ordered[output_column].to_numpy()

    df[output_column] = result

In [51]:
ROLE_EXPERIENCE_CONFIG = [
    (
        "Creator_ID_At_First_Resolution",
        "creator_previous_resolved_bugs_in_project"
    ),
    (
        "Reporter_ID_At_First_Resolution",
        "reporter_previous_resolved_bugs_in_project"
    ),
    (
        "Assignee_ID_At_First_Resolution",
        "assignee_previous_resolved_bugs_in_project"
    ),
    (
        "Resolver_ID_At_First_Resolution",
        "resolver_previous_resolutions_in_project"
    )
]


for id_column, output_column in ROLE_EXPERIENCE_CONFIG:
    add_previous_role_count_in_project(
        dataset,
        id_column=id_column,
        output_column=output_column
    )

In [52]:
dataset["creator_is_new_in_project"] = (
    dataset["has_creator"].eq(1)
    &
    dataset[
        "creator_previous_resolved_bugs_in_project"
    ].eq(0)
).astype(int)


dataset["reporter_is_new_in_project"] = (
    dataset["has_reporter"].eq(1)
    &
    dataset[
        "reporter_previous_resolved_bugs_in_project"
    ].eq(0)
).astype(int)


dataset["assignee_is_new_in_project"] = (
    dataset["has_assignee"].eq(1)
    &
    dataset[
        "assignee_previous_resolved_bugs_in_project"
    ].eq(0)
).astype(int)


dataset["resolver_is_new_in_project"] = (
    dataset["has_resolver"].eq(1)
    &
    dataset[
        "resolver_previous_resolutions_in_project"
    ].eq(0)
).astype(int)

In [53]:
experience_cols = [
    "creator_previous_resolved_bugs_in_project",
    "reporter_previous_resolved_bugs_in_project",
    "assignee_previous_resolved_bugs_in_project",
    "resolver_previous_resolutions_in_project",
    "creator_is_new_in_project",
    "reporter_is_new_in_project",
    "assignee_is_new_in_project",
    "resolver_is_new_in_project"
]

dataset[experience_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
creator_previous_resolved_bugs_in_project,67806.0,78.730747,129.991479,0.0,2.0,20.0,97.0,933.0
reporter_previous_resolved_bugs_in_project,67806.0,78.853818,130.163639,0.0,2.0,20.0,97.0,935.0
assignee_previous_resolved_bugs_in_project,67806.0,350.209568,562.179522,0.0,10.0,122.0,423.0,3528.0
resolver_previous_resolutions_in_project,67806.0,435.189467,605.194910,0.0,66.0,221.0,525.0,3592.0
creator_is_new_in_project,67806.0,0.171651,0.377080,0.0,0.0,0.0,0.0,1.0
reporter_is_new_in_project,67806.0,0.170958,0.376475,0.0,0.0,0.0,0.0,1.0
assignee_is_new_in_project,67806.0,0.012138,0.109501,0.0,0.0,0.0,0.0,1.0
resolver_is_new_in_project,67806.0,0.012359,0.110482,0.0,0.0,0.0,0.0,1.0


In [54]:
dataset_summary = pd.DataFrame({
    "metric": [
        "filas_dataset",
        "bugs_reabiertos",
        "tasa_reapertura_pct",
        "variables_actuales"
    ],
    "value": [
        len(dataset),
        dataset["Reopened_Target"].sum(),
        round(100 * dataset["Reopened_Target"].mean(), 2),
        dataset.shape[1]
    ]
})

dataset_summary

,metric,value
0,filas_dataset,67806.00
1,bugs_reabiertos,7635.00
2,tasa_reapertura_pct,11.26
3,variables_actuales,62.00


In [55]:
change_field_totals = (
    change_field_features
    .drop(columns=KEY_COLS)
    .sum()
    .sort_values(ascending=False)
)
change_field_totals.head(30)


n_change_other_until_resolution               447182
n_change_status_until_resolution              189228
n_change_fix_version_until_resolution         102205
n_change_people_until_resolution               63303
n_change_link_until_resolution                 38123
n_change_resolution_until_resolution           31825
n_change_priority_until_resolution             25372
n_change_description_until_resolution          23629
n_change_affected_version_until_resolution     23510
n_change_component_until_resolution            17261
n_change_summary_until_resolution              12363
n_change_story_point_until_resolution           5801
dtype: int64

In [56]:
# La clasificación de campos se aplicó durante la lectura por bloques.
print("Clasificación de campos: OK")


Clasificación de campos: OK


In [57]:
change_features_general.head()


,Issue_ID,Project_ID,n_changes_until_resolution,n_change_authors_until_resolution
0,27390,7,16,3
1,27392,7,6,2
2,27394,7,19,3
3,27407,7,9,2
4,27408,7,9,3


In [58]:
change_field_features.head()


,Issue_ID,Project_ID,n_change_affected_version_until_resolution,n_change_component_until_resolution,n_change_description_until_resolution,n_change_fix_version_until_resolution,n_change_link_until_resolution,n_change_other_until_resolution,n_change_people_until_resolution,n_change_priority_until_resolution,n_change_resolution_until_resolution,n_change_status_until_resolution,n_change_story_point_until_resolution,n_change_summary_until_resolution
0,27390,7,2,0,2,1,1,4,1,0,0,4,0,1
1,27392,7,0,0,1,1,0,0,0,0,0,4,0,0
2,27394,7,0,0,7,1,0,2,2,0,0,4,0,3
3,27407,7,0,0,1,1,1,1,0,1,0,4,0,0
4,27408,7,0,0,2,1,0,1,1,0,0,4,0,0


In [59]:
dataset = dataset.merge(
    change_features_general,
    on=KEY_COLS,
    how="left",
    validate="one_to_one"
)

dataset = dataset.merge(
    change_field_features,
    on=KEY_COLS,
    how="left",
    validate="one_to_one"
)

change_feature_cols = [
    col for col in dataset.columns
    if col.startswith("n_change_")
    or col == "n_changes_until_resolution"
]

dataset[change_feature_cols] = (
    dataset[change_feature_cols]
    .fillna(0)
    .astype("int64")
)

dataset["n_priority_changes_until_resolution"] = (
    dataset.get(
        "n_change_priority_until_resolution",
        pd.Series(0, index=dataset.index)
    )
    .astype("int64")
)

dataset["had_priority_change_until_resolution"] = (
    dataset["n_priority_changes_until_resolution"] > 0
).astype(int)

dataset.shape

(67806, 78)

In [60]:
dataset[change_feature_cols].describe().T.sort_index()

,count,mean,std,min,25%,50%,75%,max
n_change_affected_version_until_resolution,67806.0,0.346724,1.104499,0.0,0.0,0.0,0.0,38.0
n_change_authors_until_resolution,67806.0,3.164749,2.141821,1.0,2.0,3.0,4.0,21.0
n_change_component_until_resolution,67806.0,0.254564,0.948687,0.0,0.0,0.0,0.0,92.0
n_change_description_until_resolution,67806.0,0.348479,0.928033,0.0,0.0,0.0,0.0,35.0
n_change_fix_version_until_resolution,67806.0,1.507315,3.014553,0.0,0.0,1.0,2.0,259.0
n_change_link_until_resolution,67806.0,0.562236,1.149184,0.0,0.0,0.0,1.0,24.0
n_change_other_until_resolution,67806.0,6.595021,10.690242,0.0,0.0,2.0,8.0,173.0
n_change_people_until_resolution,67806.0,0.933590,1.184156,0.0,0.0,1.0,1.0,15.0
n_change_priority_until_resolution,67806.0,0.374185,0.611520,0.0,0.0,0.0,1.0,8.0
n_change_resolution_until_resolution,67806.0,0.469354,0.587556,0.0,0.0,0.0,1.0,11.0


## 12. Camino de estados hasta la primera resolución

Además de contar cambios generales, se construyen variables específicas del recorrido de estados del bug antes de su primera resolución.

Estas variables permiten capturar si el bug pasó por estados activos, validaciones, retornos desde validación o múltiples cambios de estado antes de considerarse resuelto.

In [61]:
status_until_resolution = status_changes_mapped.merge(
    first_resolution[
        [
            "Issue_ID",
            "Project_ID",
            "First_Resolution_Change_ID",
            "First_Resolution_Date"
        ]
    ],
    on=["Issue_ID", "Project_ID"],
    how="inner"
)

status_until_resolution = status_until_resolution[
    (
        status_until_resolution["Change_Date"] < status_until_resolution["First_Resolution_Date"]
    )
    |
    (
        (status_until_resolution["Change_Date"] == status_until_resolution["First_Resolution_Date"]) &
        (status_until_resolution["Change_ID"] <= status_until_resolution["First_Resolution_Change_ID"])
    )
].copy()

status_until_resolution.shape

(189228, 19)

In [62]:
status_path_general = (
    status_until_resolution
    .groupby(["Issue_ID", "Project_ID"])
    .agg(
        n_status_events_until_resolution=("Change_ID", "count"),
        n_distinct_to_statuses_until_resolution=("To_Status", "nunique"),
        n_distinct_to_status_groups_until_resolution=("To_Status_Group", "nunique")
    )
    .reset_index()
)

status_path_general.head()

,Issue_ID,Project_ID,n_status_events_until_resolution,n_distinct_to_statuses_until_resolution,n_distinct_to_status_groups_until_resolution
0,27390,7,4,4,3
1,27392,7,4,4,3
2,27394,7,4,4,3
3,27407,7,4,4,3
4,27408,7,4,4,3


In [63]:
status_group_features = (
    status_until_resolution
    .groupby(["Issue_ID", "Project_ID", "To_Status_Group"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

status_group_features = status_group_features.rename(
    columns={
        col: f"n_entries_{col}_until_resolution"
        for col in status_group_features.columns
        if col not in ["Issue_ID", "Project_ID"]
    }
)

status_group_features.head()

To_Status_Group,Issue_ID,Project_ID,n_entries_active_until_resolution,n_entries_explicit_reopen_until_resolution,n_entries_resolution_until_resolution,n_entries_validation_until_resolution
0,27390,7,1,0,1,2
1,27392,7,1,0,1,2
2,27394,7,1,0,1,2
3,27407,7,1,0,1,2
4,27408,7,1,0,1,2


In [64]:
status_until_resolution["transition_group"] = (
    status_until_resolution["From_Status_Group"].fillna("missing")
    + "_to_"
    + status_until_resolution["To_Status_Group"].fillna("missing")
)

transition_features = (
    status_until_resolution[
        status_until_resolution["transition_group"].isin([
            "active_to_validation",
            "validation_to_active",
            "active_to_resolution",
            "validation_to_resolution",
            "active_to_explicit_reopen",
            "resolution_to_active",
            "resolution_to_explicit_reopen",
        ])
    ]
    .groupby(["Issue_ID", "Project_ID", "transition_group"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

transition_features = transition_features.rename(
    columns={
        col: f"n_transition_{col}_until_resolution"
        for col in transition_features.columns
        if col not in ["Issue_ID", "Project_ID"]
    }
)

transition_features.head()

transition_group,Issue_ID,Project_ID,n_transition_active_to_explicit_reopen_until_resolution,n_transition_active_to_resolution_until_resolution,n_transition_active_to_validation_until_resolution,n_transition_resolution_to_explicit_reopen_until_resolution,n_transition_validation_to_active_until_resolution,n_transition_validation_to_resolution_until_resolution
0,27390,7,0,0,1,0,0,1
1,27392,7,0,0,1,0,0,1
2,27394,7,0,0,1,0,0,1
3,27407,7,0,0,1,0,0,1
4,27408,7,0,0,1,0,0,1


In [65]:
dataset = dataset.merge(
    status_path_general,
    on=["Issue_ID", "Project_ID"],
    how="left"
)

dataset = dataset.merge(
    status_group_features,
    on=["Issue_ID", "Project_ID"],
    how="left"
)

dataset = dataset.merge(
    transition_features,
    on=["Issue_ID", "Project_ID"],
    how="left"
)

status_feature_cols = [
    col for col in dataset.columns
    if col.startswith("n_status_")
    or col.startswith("n_distinct_to_status")
    or col.startswith("n_entries_")
    or col.startswith("n_transition_")
]

dataset[status_feature_cols] = dataset[status_feature_cols].fillna(0)

dataset.shape

dataset.shape

(67806, 91)

In [66]:
dataset[
    [
        "Status_Before_First_Resolution",
        "Status_Group_Before_First_Resolution"
    ]
].isna().sum()

Status_Before_First_Resolution          0
Status_Group_Before_First_Resolution    0
dtype: int64

In [67]:
def feature_or_zero(column_name):
    if column_name in dataset.columns:
        return dataset[column_name]
    return pd.Series(0, index=dataset.index, dtype="int64")


dataset["had_validation_until_resolution"] = (
    feature_or_zero("n_entries_validation_until_resolution") > 0
).astype(int)

dataset["had_active_return_from_validation_until_resolution"] = (
    feature_or_zero("n_transition_validation_to_active_until_resolution") > 0
).astype(int)

dataset["had_reopen_like_transition_until_resolution"] = (
    (
        feature_or_zero("n_transition_resolution_to_active_until_resolution")
        +
        feature_or_zero("n_transition_resolution_to_explicit_reopen_until_resolution")
        +
        feature_or_zero("n_entries_explicit_reopen_until_resolution")
    ) > 0
).astype(int)

dataset[
    [
        "had_validation_until_resolution",
        "had_active_return_from_validation_until_resolution",
        "had_reopen_like_transition_until_resolution"
    ]
].mean()

had_validation_until_resolution                       0.243813
had_active_return_from_validation_until_resolution    0.087278
had_reopen_like_transition_until_resolution           0.024673
dtype: float64

In [68]:
status_path_summary = dataset[
    [
        "n_status_events_until_resolution",
        "n_distinct_to_statuses_until_resolution",
        "n_distinct_to_status_groups_until_resolution",
        "had_validation_until_resolution",
        "had_active_return_from_validation_until_resolution",
        "had_reopen_like_transition_until_resolution"
    ]
].describe()

status_path_summary

,n_status_events_until_resolution,n_distinct_to_statuses_until_resolution,n_distinct_to_status_groups_until_resolution,had_validation_until_resolution,had_active_return_from_validation_until_resolution,had_reopen_like_transition_until_resolution
count,67806.000000,67806.000000,67806.000000,67806.000000,67806.000000,67806.000000
mean,2.790726,2.432322,1.631817,0.243813,0.087278,0.024673
std,3.438429,2.380695,0.824298,0.429385,0.282245,0.155129
min,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
50%,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,3.000000,3.000000,2.000000,0.000000,0.000000,0.000000
max,61.000000,13.000000,4.000000,1.000000,1.000000,1.000000


## 13. Comentarios hasta la primera resolución

Se construyen variables a partir de los comentarios registrados antes o hasta el momento de la primera resolución del bug.

Estas variables buscan capturar el nivel de discusión, participación y detalle asociado al bug antes de que fuera considerado resuelto por primera vez.

In [69]:
# La tabla comment contiene más de un millón de filas y campos de texto
# potencialmente extensos. No se cargan los textos completos en pandas.
# MySQL calcula las longitudes y pandas procesa el resultado por bloques.

import gc

COMMENT_CHUNK_SIZE = 100_000

comment_query = f"""
SELECT
    c.ID AS Comment_ID,
    c.Issue_ID,
    i.Project_ID,
    c.Creation_Date AS Comment_Date,
    c.Author_ID,
    CHAR_LENGTH(
        TRIM(COALESCE(c.Comment_Text, ''))
    ) AS comment_text_length,
    CHAR_LENGTH(
        TRIM(COALESCE(c.Comment_Code, ''))
    ) AS comment_code_length
FROM `comment` c
JOIN `issue` i
    ON i.ID = c.Issue_ID
WHERE i.Type = 'Bug'
  AND i.Project_ID IN ({project_ids_sql})
ORDER BY c.ID
"""

comment_resolution_lookup = (
    first_resolution[
        [
            "Issue_ID",
            "Project_ID",
            "First_Resolution_Date",
        ]
    ]
    .drop_duplicates(["Issue_ID", "Project_ID"])
)

comment_summary_parts = []
comment_author_parts = []

comment_source_rows = 0
comments_until_resolution_rows = 0
comments_until_resolution_temporal_ok = True

with engine.connect().execution_options(
    stream_results=True,
    max_row_buffer=COMMENT_CHUNK_SIZE,
) as connection:
    comment_chunks = pd.read_sql_query(
        comment_query,
        connection,
        parse_dates=["Comment_Date"],
        chunksize=COMMENT_CHUNK_SIZE,
    )

    for chunk_number, comment_chunk in enumerate(
        comment_chunks,
        start=1,
    ):
        comment_source_rows += len(comment_chunk)

        # Solo se conservan comentarios de bugs que poseen una primera
        # resolución identificada.
        comment_chunk = comment_chunk.merge(
            comment_resolution_lookup,
            on=["Issue_ID", "Project_ID"],
            how="inner",
            validate="many_to_one",
        )

        # Se descartan inmediatamente los comentarios posteriores a la
        # primera resolución.
        comment_chunk = comment_chunk.loc[
            comment_chunk["Comment_Date"].le(
                comment_chunk["First_Resolution_Date"]
            )
        ].copy()

        comments_until_resolution_rows += len(comment_chunk)

        if comment_chunk.empty:
            del comment_chunk
            gc.collect()
            continue

        if not (
            comment_chunk["Comment_Date"]
            <= comment_chunk["First_Resolution_Date"]
        ).all():
            comments_until_resolution_temporal_ok = False
            raise RuntimeError(
                "Se detectaron comentarios posteriores a la "
                "primera resolución."
            )

        # Las longitudes ya fueron calculadas en MySQL. Se convierten a
        # tipos compactos y no se conservan los textos.
        comment_chunk["comment_text_length"] = (
            pd.to_numeric(
                comment_chunk["comment_text_length"],
                errors="coerce",
            )
            .fillna(0)
            .astype("int32")
        )

        comment_chunk["comment_code_length"] = (
            pd.to_numeric(
                comment_chunk["comment_code_length"],
                errors="coerce",
            )
            .fillna(0)
            .astype("int32")
        )

        comment_chunk["has_comment_text"] = (
            comment_chunk["comment_text_length"].gt(0)
        ).astype("int8")

        comment_chunk["has_comment_code"] = (
            comment_chunk["comment_code_length"].gt(0)
        ).astype("int8")

        # Resumen parcial por issue. Las sumas, máximos y fechas pueden
        # combinarse exactamente entre bloques.
        chunk_summary = (
            comment_chunk
            .groupby(
                ["Issue_ID", "Project_ID"],
                sort=False,
                observed=True,
            )
            .agg(
                n_comments_until_resolution=(
                    "Comment_ID",
                    "count",
                ),
                comment_text_total_length_until_resolution=(
                    "comment_text_length",
                    "sum",
                ),
                comment_text_max_length_until_resolution=(
                    "comment_text_length",
                    "max",
                ),
                comment_code_total_length_until_resolution=(
                    "comment_code_length",
                    "sum",
                ),
                comment_code_max_length_until_resolution=(
                    "comment_code_length",
                    "max",
                ),
                has_comment_text_until_resolution=(
                    "has_comment_text",
                    "max",
                ),
                has_comment_code_until_resolution=(
                    "has_comment_code",
                    "max",
                ),
                First_Comment_Date=(
                    "Comment_Date",
                    "min",
                ),
                Last_Comment_Date=(
                    "Comment_Date",
                    "max",
                ),
            )
            .reset_index()
        )

        comment_summary_parts.append(chunk_summary)

        # Para obtener el número exacto de autores diferentes se guardan
        # únicamente las ternas issue-proyecto-autor, sin textos ni fechas.
        chunk_authors = (
            comment_chunk.loc[
                comment_chunk["Author_ID"].notna(),
                ["Issue_ID", "Project_ID", "Author_ID"],
            ]
            .drop_duplicates()
        )

        if not chunk_authors.empty:
            comment_author_parts.append(chunk_authors)

        if chunk_number % 5 == 0:
            print(
                f"Bloques de comentarios: {chunk_number} | "
                f"filas fuente: {comment_source_rows:,} | "
                "filas hasta primera resolución: "
                f"{comments_until_resolution_rows:,}"
            )

        del comment_chunk
        del chunk_summary
        del chunk_authors
        gc.collect()


if comment_summary_parts:
    comment_summary_all = pd.concat(
        comment_summary_parts,
        ignore_index=True,
    )

    comment_features = (
        comment_summary_all
        .groupby(
            ["Issue_ID", "Project_ID"],
            sort=False,
            observed=True,
        )
        .agg(
            n_comments_until_resolution=(
                "n_comments_until_resolution",
                "sum",
            ),
            comment_text_total_length_until_resolution=(
                "comment_text_total_length_until_resolution",
                "sum",
            ),
            comment_text_max_length_until_resolution=(
                "comment_text_max_length_until_resolution",
                "max",
            ),
            comment_code_total_length_until_resolution=(
                "comment_code_total_length_until_resolution",
                "sum",
            ),
            comment_code_max_length_until_resolution=(
                "comment_code_max_length_until_resolution",
                "max",
            ),
            has_comment_text_until_resolution=(
                "has_comment_text_until_resolution",
                "max",
            ),
            has_comment_code_until_resolution=(
                "has_comment_code_until_resolution",
                "max",
            ),
            First_Comment_Date=(
                "First_Comment_Date",
                "min",
            ),
            Last_Comment_Date=(
                "Last_Comment_Date",
                "max",
            ),
        )
        .reset_index()
    )
else:
    comment_features = pd.DataFrame(
        columns=[
            "Issue_ID",
            "Project_ID",
            "n_comments_until_resolution",
            "comment_text_total_length_until_resolution",
            "comment_text_max_length_until_resolution",
            "comment_code_total_length_until_resolution",
            "comment_code_max_length_until_resolution",
            "has_comment_text_until_resolution",
            "has_comment_code_until_resolution",
            "First_Comment_Date",
            "Last_Comment_Date",
        ]
    )


# Cantidad exacta de autores distintos por issue.
if comment_author_parts:
    comment_authors = (
        pd.concat(
            comment_author_parts,
            ignore_index=True,
        )
        .drop_duplicates(
            ["Issue_ID", "Project_ID", "Author_ID"]
        )
    )

    comment_author_features = (
        comment_authors
        .groupby(
            ["Issue_ID", "Project_ID"],
            sort=False,
            observed=True,
        )
        .size()
        .rename("n_comment_authors_until_resolution")
        .reset_index()
    )

    comment_features = comment_features.merge(
        comment_author_features,
        on=["Issue_ID", "Project_ID"],
        how="left",
        validate="one_to_one",
    )
else:
    comment_features[
        "n_comment_authors_until_resolution"
    ] = 0


comment_features[
    "n_comment_authors_until_resolution"
] = (
    comment_features[
        "n_comment_authors_until_resolution"
    ]
    .fillna(0)
    .astype("int32")
)

# El promedio original se obtiene exactamente como suma de longitudes
# dividida por cantidad de comentarios, incluyendo longitud cero.
comment_features[
    "comment_text_avg_length_until_resolution"
] = np.divide(
    comment_features[
        "comment_text_total_length_until_resolution"
    ],
    comment_features["n_comments_until_resolution"],
    out=np.zeros(
        len(comment_features),
        dtype=float,
    ),
    where=comment_features[
        "n_comments_until_resolution"
    ].to_numpy() > 0,
)

comment_features[
    "comment_code_avg_length_until_resolution"
] = np.divide(
    comment_features[
        "comment_code_total_length_until_resolution"
    ],
    comment_features["n_comments_until_resolution"],
    out=np.zeros(
        len(comment_features),
        dtype=float,
    ),
    where=comment_features[
        "n_comments_until_resolution"
    ].to_numpy() > 0,
)

# Liberación explícita de objetos auxiliares.
del comment_summary_parts
del comment_author_parts

if "comment_summary_all" in globals():
    del comment_summary_all

if "comment_authors" in globals():
    del comment_authors

if "comment_author_features" in globals():
    del comment_author_features

gc.collect()

print(
    "Filas fuente de comment:",
    f"{comment_source_rows:,}",
)
print(
    "Comentarios hasta primera resolución:",
    f"{comments_until_resolution_rows:,}",
)
print(
    "Issues con comentarios previos:",
    f"{len(comment_features):,}",
)

comment_features.shape


Bloques de comentarios: 5 | filas fuente: 410,327 | filas hasta primera resolución: 295,362
Filas fuente de comment: 410,327
Comentarios hasta primera resolución: 295,362
Issues con comentarios previos: 62,755


(62755, 14)

In [70]:
# El filtrado temporal de comentarios ya se realizó por bloques
# en la celda anterior. No se materializa comments_until_resolution.

comment_features.head()


,Issue_ID,Project_ID,n_comments_until_resolution,comment_text_total_length_until_resolution,comment_text_max_length_until_resolution,comment_code_total_length_until_resolution,comment_code_max_length_until_resolution,has_comment_text_until_resolution,has_comment_code_until_resolution,First_Comment_Date,Last_Comment_Date,n_comment_authors_until_resolution,comment_text_avg_length_until_resolution,comment_code_avg_length_until_resolution
0,27390,7,5,1058,457,0,0,1,0,2020-08-28 11:53:18,2020-09-14 19:26:38,3,211.600000,0.0
1,27392,7,3,582,309,0,0,1,0,2020-08-07 15:46:12,2020-08-13 15:39:39,2,194.000000,0.0
2,27394,7,6,1893,635,276,276,1,1,2020-07-30 22:09:06,2020-08-06 13:56:36,3,315.500000,46.0
3,27407,7,3,641,502,0,0,1,0,2020-02-20 19:08:57,2020-02-25 18:37:09,2,213.666667,0.0
4,27408,7,3,500,302,0,0,1,0,2020-06-30 14:38:46,2020-08-13 16:35:56,2,166.666667,0.0


In [71]:
# Las longitudes y los indicadores de contenido fueron calculados
# durante la lectura por bloques, sin cargar los textos completos.

comment_features[
    [
        "n_comments_until_resolution",
        "n_comment_authors_until_resolution",
        "comment_text_total_length_until_resolution",
        "comment_text_avg_length_until_resolution",
        "comment_text_max_length_until_resolution",
        "comment_code_total_length_until_resolution",
        "comment_code_avg_length_until_resolution",
        "comment_code_max_length_until_resolution",
    ]
].describe().T


,count,mean,std,min,25%,50%,75%,max
n_comments_until_resolution,62755.0,4.706589,6.757116,1.0,1.0,2.00,5.000000,124.0
n_comment_authors_until_resolution,62755.0,2.539893,2.079019,0.0,1.0,2.00,3.000000,39.0
comment_text_total_length_until_resolution,62755.0,1811.634499,6419.447314,3.0,144.0,440.00,1283.000000,916178.0
comment_text_avg_length_until_resolution,62755.0,264.178681,1927.566747,3.0,91.5,176.75,304.083333,458089.0
comment_text_max_length_until_resolution,62755.0,644.546411,4302.797296,3.0,114.0,274.00,578.000000,915960.0
comment_code_total_length_until_resolution,62755.0,250.557804,14220.756956,0.0,0.0,0.00,0.000000,3311851.0
comment_code_avg_length_until_resolution,62755.0,34.513487,853.192212,0.0,0.0,0.00,0.000000,106142.0
comment_code_max_length_until_resolution,62755.0,221.358394,14029.664159,0.0,0.0,0.00,0.000000,3271018.0


In [72]:
# comment_features ya quedó agregado a una fila por issue.
assert not comment_features.duplicated(
    ["Issue_ID", "Project_ID"]
).any()

comment_features.head()


,Issue_ID,Project_ID,n_comments_until_resolution,comment_text_total_length_until_resolution,comment_text_max_length_until_resolution,comment_code_total_length_until_resolution,comment_code_max_length_until_resolution,has_comment_text_until_resolution,has_comment_code_until_resolution,First_Comment_Date,Last_Comment_Date,n_comment_authors_until_resolution,comment_text_avg_length_until_resolution,comment_code_avg_length_until_resolution
0,27390,7,5,1058,457,0,0,1,0,2020-08-28 11:53:18,2020-09-14 19:26:38,3,211.600000,0.0
1,27392,7,3,582,309,0,0,1,0,2020-08-07 15:46:12,2020-08-13 15:39:39,2,194.000000,0.0
2,27394,7,6,1893,635,276,276,1,1,2020-07-30 22:09:06,2020-08-06 13:56:36,3,315.500000,46.0
3,27407,7,3,641,502,0,0,1,0,2020-02-20 19:08:57,2020-02-25 18:37:09,2,213.666667,0.0
4,27408,7,3,500,302,0,0,1,0,2020-06-30 14:38:46,2020-08-13 16:35:56,2,166.666667,0.0


In [73]:
comment_features = comment_features.merge(
    first_resolution[
        [
            "Issue_ID",
            "Project_ID",
            "First_Resolution_Date"
        ]
    ],
    on=["Issue_ID", "Project_ID"],
    how="left"
)

comment_features["hours_first_comment_to_resolution"] = (
    comment_features["First_Resolution_Date"] - comment_features["First_Comment_Date"]
).dt.total_seconds() / 3600

comment_features["hours_last_comment_to_resolution"] = (
    comment_features["First_Resolution_Date"] - comment_features["Last_Comment_Date"]
).dt.total_seconds() / 3600

comment_features = comment_features.drop(columns=["First_Resolution_Date"])

comment_features.head()

,Issue_ID,Project_ID,n_comments_until_resolution,comment_text_total_length_until_resolution,comment_text_max_length_until_resolution,comment_code_total_length_until_resolution,comment_code_max_length_until_resolution,has_comment_text_until_resolution,has_comment_code_until_resolution,First_Comment_Date,Last_Comment_Date,n_comment_authors_until_resolution,comment_text_avg_length_until_resolution,comment_code_avg_length_until_resolution,hours_first_comment_to_resolution,hours_last_comment_to_resolution
0,27390,7,5,1058,457,0,0,1,0,2020-08-28 11:53:18,2020-09-14 19:26:38,3,211.600000,0.0,415.562222,0.006667
1,27392,7,3,582,309,0,0,1,0,2020-08-07 15:46:12,2020-08-13 15:39:39,2,194.000000,0.0,143.892222,0.001389
2,27394,7,6,1893,635,276,276,1,1,2020-07-30 22:09:06,2020-08-06 13:56:36,3,315.500000,46.0,287.250556,127.458889
3,27407,7,3,641,502,0,0,1,0,2020-02-20 19:08:57,2020-02-25 18:37:09,2,213.666667,0.0,119.471111,0.001111
4,27408,7,3,500,302,0,0,1,0,2020-06-30 14:38:46,2020-08-13 16:35:56,2,166.666667,0.0,1057.953889,0.001111


In [74]:
dataset = dataset.merge(
    comment_features,
    on=["Issue_ID", "Project_ID"],
    how="left"
)

comment_count_cols = [
    "n_comments_until_resolution",
    "n_comment_authors_until_resolution",
    "comment_text_total_length_until_resolution",
    "comment_text_avg_length_until_resolution",
    "comment_text_max_length_until_resolution",
    "comment_code_total_length_until_resolution",
    "comment_code_avg_length_until_resolution",
    "comment_code_max_length_until_resolution",
    "has_comment_text_until_resolution",
    "has_comment_code_until_resolution"
]

dataset[comment_count_cols] = dataset[comment_count_cols].fillna(0)

dataset.shape

(67806, 108)

In [75]:
dataset[
    [
        "n_comments_until_resolution",
        "n_comment_authors_until_resolution",
        "comment_text_total_length_until_resolution",
        "comment_text_avg_length_until_resolution",
        "comment_text_max_length_until_resolution",
        "comment_code_total_length_until_resolution",
        "comment_code_avg_length_until_resolution",
        "comment_code_max_length_until_resolution",
        "has_comment_text_until_resolution",
        "has_comment_code_until_resolution",
        "hours_first_comment_to_resolution",
        "hours_last_comment_to_resolution"
    ]
].describe()

,n_comments_until_resolution,n_comment_authors_until_resolution,comment_text_total_length_until_resolution,comment_text_avg_length_until_resolution,comment_text_max_length_until_resolution,comment_code_total_length_until_resolution,comment_code_avg_length_until_resolution,comment_code_max_length_until_resolution,has_comment_text_until_resolution,has_comment_code_until_resolution,hours_first_comment_to_resolution,hours_last_comment_to_resolution
count,67806.000000,67806.000000,67806.000000,67806.000000,67806.000000,6.780600e+04,67806.000000,6.780600e+04,67806.000000,67806.000000,62755.000000,62755.000000
mean,4.355986,2.350692,1676.682344,244.499500,596.532903,2.318933e+02,31.942510,2.048690e+02,0.925508,0.104209,4692.581952,947.048370
std,6.616994,2.108340,6194.011066,1855.679207,4142.890034,1.368099e+04,820.848856,1.349712e+04,0.262572,0.305534,11905.750226,6231.748704
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,104.000000,73.500000,85.000000,0.000000e+00,0.000000,0.000000e+00,1.000000,0.000000,0.000278,0.000000
50%,2.000000,2.000000,368.000000,162.408333,242.000000,0.000000e+00,0.000000,0.000000e+00,1.000000,0.000000,101.646389,0.000000
75%,5.000000,3.000000,1149.000000,290.307692,542.000000,0.000000e+00,0.000000,0.000000e+00,1.000000,0.000000,1444.166806,0.002500
max,124.000000,39.000000,916178.000000,458089.000000,915960.000000,3.311851e+06,106142.000000,3.271018e+06,1.000000,1.000000,118934.775278,82110.547500


## 14. Componentes, versiones y vínculos reconstruidos en la primera resolución

Las tablas relacionales contienen el estado final del issue. Para evitar fuga temporal,
se parte de ese estado y se deshacen, en orden cronológico inverso, todos los cambios
posteriores a la primera resolución registrados en `change_log`.

Los identificadores internos de TAWOS se convierten a `Jira_ID`, porque los valores
de `change_log` corresponden a los identificadores de Jira.

In [76]:
def get_table_columns(table_name):
    return set(
        pd.read_sql(
            f"SHOW COLUMNS FROM `{table_name}`",
            engine
        )["Field"]
        .astype(str)
        .tolist()
    )


fix_version_columns = get_table_columns("fix_version")

if "Fix_Version_ID" in fix_version_columns:
    fix_version_fk = "Fix_Version_ID"
elif "Fixed_Version_ID" in fix_version_columns:
    fix_version_fk = "Fixed_Version_ID"
else:
    raise KeyError(
        "No se encontró la columna de versión corregida en fix_version"
    )


issue_components_current = pd.read_sql(
    f"""
    SELECT
        ic.Issue_ID,
        i.Project_ID,
        COALESCE(
            CAST(c.Jira_ID AS CHAR),
            CONCAT('internal:', CAST(c.ID AS CHAR))
        ) AS Entity_Token
    FROM issue_component ic
    JOIN `issue` i
        ON i.ID = ic.Issue_ID
    JOIN component c
        ON c.ID = ic.Component_ID
    WHERE i.Type = 'Bug'
      AND i.Project_ID IN ({project_ids_sql})
    """,
    engine
)


affected_versions_current = pd.read_sql(
    f"""
    SELECT
        av.Issue_ID,
        i.Project_ID,
        COALESCE(
            CAST(v.Jira_ID AS CHAR),
            CONCAT('internal:', CAST(v.ID AS CHAR))
        ) AS Entity_Token
    FROM affected_version av
    JOIN `issue` i
        ON i.ID = av.Issue_ID
    JOIN `version` v
        ON v.ID = av.Affected_Version_ID
    WHERE i.Type = 'Bug'
      AND i.Project_ID IN ({project_ids_sql})
    """,
    engine
)


fix_versions_current = pd.read_sql(
    f"""
    SELECT
        fv.Issue_ID,
        i.Project_ID,
        COALESCE(
            CAST(v.Jira_ID AS CHAR),
            CONCAT('internal:', CAST(v.ID AS CHAR))
        ) AS Entity_Token
    FROM fix_version fv
    JOIN `issue` i
        ON i.ID = fv.Issue_ID
    JOIN `version` v
        ON v.ID = fv.`{fix_version_fk}`
    WHERE i.Type = 'Bug'
      AND i.Project_ID IN ({project_ids_sql})
    """,
    engine
)


# Se reconstruyen los vínculos cuyo registro pertenece al issue
# (issue_link.Issue_ID), que es el lado atribuible a su change_log.
issue_links_current = pd.read_sql(
    f"""
    SELECT
        il.Issue_ID,
        source_issue.Project_ID,
        COALESCE(
            CAST(target_issue.Jira_ID AS CHAR),
            target_issue.Issue_Key,
            CONCAT('internal:', CAST(target_issue.ID AS CHAR))
        ) AS Entity_Token
    FROM issue_link il
    JOIN `issue` source_issue
        ON source_issue.ID = il.Issue_ID
    JOIN `issue` target_issue
        ON target_issue.ID = il.Target_Issue_ID
    WHERE source_issue.Type = 'Bug'
      AND source_issue.Project_ID IN ({project_ids_sql})
    """,
    engine
)

(
    issue_components_current.shape,
    affected_versions_current.shape,
    fix_versions_current.shape,
    issue_links_current.shape
)

((87245, 3), (94042, 3), (62621, 3), (41115, 3))

In [77]:
SET_SPLIT_PATTERN = re.compile(r"\s*(?:,|;|\||\r?\n)\s*")


def normalize_entity_token(value):
    if pd.isna(value):
        return None

    text_value = str(value).strip()

    if text_value.casefold() in NULL_TEXT_VALUES:
        return None

    # Evita que identificadores numéricos aparezcan como 123.0
    if re.fullmatch(r"-?\d+(?:\.0+)?", text_value):
        return str(int(float(text_value)))

    return text_value.casefold()


def parse_entity_tokens(value):
    if pd.isna(value):
        return set()

    text_value = str(value).strip()

    if text_value.casefold() in NULL_TEXT_VALUES:
        return set()

    # Algunos dumps guardan listas serializadas.
    if (
        (text_value.startswith("[") and text_value.endswith("]"))
        or
        (text_value.startswith("(") and text_value.endswith(")"))
    ):
        try:
            parsed = ast.literal_eval(text_value)
            if isinstance(parsed, (list, tuple, set)):
                return {
                    token
                    for item in parsed
                    if (token := normalize_entity_token(item)) is not None
                }
        except (ValueError, SyntaxError):
            pass

    text_value = text_value.strip("[](){}")

    parts = SET_SPLIT_PATTERN.split(text_value)

    return {
        token
        for part in parts
        if (token := normalize_entity_token(part)) is not None
    }


def tokens_from_change_side(row, side):
    value_tokens = parse_entity_tokens(row[f"{side}_Value"])

    if value_tokens:
        return value_tokens, "value"

    string_tokens = parse_entity_tokens(row[f"{side}_String"])

    if string_tokens:
        return string_tokens, "string"

    return set(), "empty"


def current_set_map(current_members):
    result = {
        (int(issue_id), int(project_id)): set()
        for issue_id, project_id in bug_target[KEY_COLS].itertuples(
            index=False,
            name=None
        )
    }

    for row in current_members.itertuples(index=False):
        key = (int(row.Issue_ID), int(row.Project_ID))
        token = normalize_entity_token(row.Entity_Token)

        if token is not None and key in result:
            result[key].add(token)

    return result


def reconstruct_set_at_resolution(
    current_members,
    field_names,
    count_column,
    flag_column
):
    """
    Parte del conjunto final y revierte los cambios posteriores a la primera
    resolución desde el más reciente hasta el más antiguo.

    Para cada evento:
      - lo agregado hacia adelante (To - From) se elimina al retroceder;
      - lo eliminado hacia adelante (From - To) se vuelve a agregar.
    """

    if isinstance(field_names, str):
        field_names = [field_names]

    aliases = {
        str(name).strip().casefold()
        for name in field_names
    }

    normalized_field = (
        reconstruction_changes["Field"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    field_changes = reconstruction_changes[
        normalized_field.isin(aliases)
    ].copy()

    field_changes = field_changes.merge(
        resolution_cutoff,
        on=KEY_COLS,
        how="inner",
        validate="many_to_one"
    )

    changes_after = field_changes[
        is_after_first_resolution(field_changes)
    ].copy()

    changes_after = changes_after.sort_values(
        KEY_COLS + ["Change_Date", "Change_ID"],
        ascending=[True, True, False, False]
    )

    states = current_set_map(current_members)

    unmatched_reverse_removals = 0
    string_fallback_sides = 0

    for _, row in changes_after.iterrows():
        key = (int(row["Issue_ID"]), int(row["Project_ID"]))
        state = states[key]

        from_tokens, from_source = tokens_from_change_side(row, "From")
        to_tokens, to_source = tokens_from_change_side(row, "To")

        string_fallback_sides += int(from_source == "string")
        string_fallback_sides += int(to_source == "string")

        added_forward = to_tokens - from_tokens
        removed_forward = from_tokens - to_tokens

        unmatched_reverse_removals += len(
            added_forward - state
        )

        state.difference_update(added_forward)
        state.update(removed_forward)

    result = bug_target[KEY_COLS].copy()

    result[count_column] = [
        len(states[(int(issue_id), int(project_id))])
        for issue_id, project_id in result[KEY_COLS].itertuples(
            index=False,
            name=None
        )
    ]

    result[flag_column] = (
        result[count_column] > 0
    ).astype(int)

    audit = {
        "field_names": ", ".join(field_names),
        "post_resolution_changes_reversed": len(changes_after),
        "string_fallback_sides": string_fallback_sides,
        "unmatched_reverse_removals": unmatched_reverse_removals
    }

    return result, audit


In [78]:
component_features, component_audit = reconstruct_set_at_resolution(
    current_members=issue_components_current,
    field_names=["Component", "Components"],
    count_column="n_components_at_first_resolution",
    flag_column="has_component_at_first_resolution"
)

affected_version_features, affected_version_audit = reconstruct_set_at_resolution(
    current_members=affected_versions_current,
    field_names=["Version", "Affected Version", "Affected Versions"],
    count_column="n_affected_versions_at_first_resolution",
    flag_column="has_affected_version_at_first_resolution"
)

fix_version_features, fix_version_audit = reconstruct_set_at_resolution(
    current_members=fix_versions_current,
    field_names=["Fix Version", "Fix Versions"],
    count_column="n_fix_versions_at_first_resolution",
    flag_column="has_fix_version_at_first_resolution"
)

link_features, link_audit = reconstruct_set_at_resolution(
    current_members=issue_links_current,
    field_names=["Link", "Links"],
    count_column="n_issue_links_at_first_resolution",
    flag_column="has_issue_links_at_first_resolution"
)

structural_audit = pd.DataFrame(
    [
        component_audit,
        affected_version_audit,
        fix_version_audit,
        link_audit
    ]
)

structural_audit

,field_names,post_resolution_changes_reversed,string_fallback_sides,unmatched_reverse_removals
0,"Component, Components",3517,0,0
1,"Version, Affected Version, Affected Versions",3983,0,1
2,"Fix Version, Fix Versions",35896,0,47
3,"Link, Links",14259,0,12166


In [79]:
for feature_table in [
    component_features,
    affected_version_features,
    fix_version_features,
    link_features
]:
    dataset = dataset.merge(
        feature_table,
        on=KEY_COLS,
        how="left",
        validate="one_to_one"
    )


structural_feature_cols = [
    "n_components_at_first_resolution",
    "has_component_at_first_resolution",
    "n_affected_versions_at_first_resolution",
    "has_affected_version_at_first_resolution",
    "n_fix_versions_at_first_resolution",
    "has_fix_version_at_first_resolution",
    "n_issue_links_at_first_resolution",
    "has_issue_links_at_first_resolution"
]

dataset[structural_feature_cols] = (
    dataset[structural_feature_cols]
    .fillna(0)
    .astype("int64")
)

dataset[structural_feature_cols].describe()

,n_components_at_first_resolution,has_component_at_first_resolution,n_affected_versions_at_first_resolution,has_affected_version_at_first_resolution,n_fix_versions_at_first_resolution,has_fix_version_at_first_resolution,n_issue_links_at_first_resolution,has_issue_links_at_first_resolution
count,67806.000000,67806.000000,67806.000000,67806.000000,67806.000000,67806.000000,67806.000000,67806.000000
mean,1.078990,0.920730,1.158275,0.822538,0.805017,0.561337,0.550423,0.348406
std,0.547041,0.270162,1.156811,0.382062,0.887567,0.496227,1.024406,0.476469
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,12.000000,1.000000,39.000000,1.000000,14.000000,1.000000,28.000000,1.000000


In [80]:
# Un valor distinto de cero puede indicar que el formato de From/To no coincide
# con los Jira_ID usados para reconstruir el conjunto. No se ignora silenciosamente.
problematic_structural_reconstruction = structural_audit[
    structural_audit["unmatched_reverse_removals"] > 0
]

display(problematic_structural_reconstruction)

if not problematic_structural_reconstruction.empty:
    print(
        "ADVERTENCIA: revisar ejemplos de change_log para esos campos antes de modelar."
    )

,field_names,post_resolution_changes_reversed,string_fallback_sides,unmatched_reverse_removals
1,"Version, Affected Version, Affected Versions",3983,0,1
2,"Fix Version, Fix Versions",35896,0,47
3,"Link, Links",14259,0,12166


ADVERTENCIA: revisar ejemplos de change_log para esos campos antes de modelar.


## 15. Variables temporales e históricas observables

In [81]:
# ============================================================
# VARIABLES TEMPORALES E HISTÓRICAS SIN FUGA DE INFORMACIÓN
# ============================================================

TEMPORAL_HISTORY_HORIZON_DAYS = 180
PROJECT_ACTIVITY_WINDOWS_DAYS = (30, 90, 180)
PROJECT_CONTEXT_WINDOWS_DAYS = (90, 180)
USER_RATE_SMOOTHING_STRENGTH = 10.0

columns_before_temporal_engineering = set(dataset.columns)

# Target auxiliar exclusivamente para construir historia observable.
# No se utiliza como predictor ni se conserva en los datasets finales.
days_to_first_reopen = (
    dataset["First_Reopen_Date"]
    - dataset["First_Resolution_Date"]
).dt.total_seconds() / 86400

dataset["_target_h180_for_history"] = (
    days_to_first_reopen
    .between(
        0,
        TEMPORAL_HISTORY_HORIZON_DAYS,
        inclusive="both",
    )
    .fillna(False)
    .astype("int8")
)

dataset["_target_h180_maturity_date"] = (
    dataset["First_Resolution_Date"]
    + pd.Timedelta(days=TEMPORAL_HISTORY_HORIZON_DAYS)
)

# Estas columnas deben quedar enteras al final del notebook.
TEMPORAL_INTEGER_FEATURE_COLUMNS = [
    "resolution_year",
    "resolution_month",
    "resolution_quarter",
    "resolution_weekday",
]


# ------------------------------------------------------------
# 1. Actividad y contexto reciente del proyecto
# ------------------------------------------------------------

project_context_metric_map = {
    "resolution_time_hours": "resolution_time_hours",
    "n_comments_until_resolution": "comments_until_resolution",
    "n_status_events_until_resolution": (
        "status_events_until_resolution"
    ),
    "n_changes_until_resolution": "changes_until_resolution",
}

missing_context_metrics = sorted(
    set(project_context_metric_map) - set(dataset.columns)
)
if missing_context_metrics:
    raise ValueError(
        "Faltan variables necesarias para construir el contexto "
        f"temporal del proyecto: {missing_context_metrics}"
    )

for window_days in PROJECT_ACTIVITY_WINDOWS_DAYS:
    resolved_col = f"project_resolved_last_{window_days}d"
    reopen_event_col = (
        f"project_reopen_events_last_{window_days}d"
    )

    dataset[resolved_col] = 0
    dataset[reopen_event_col] = 0

    TEMPORAL_INTEGER_FEATURE_COLUMNS.extend(
        [resolved_col, reopen_event_col]
    )

for window_days in PROJECT_CONTEXT_WINDOWS_DAYS:
    for output_suffix in project_context_metric_map.values():
        dataset[
            f"project_median_{output_suffix}_last_{window_days}d"
        ] = np.nan

    matured_col = f"project_matured_bugs_last_{window_days}d"
    reopened_col = (
        "project_reopened_within_180d_"
        f"matured_last_{window_days}d"
    )

    dataset[matured_col] = 0
    dataset[reopened_col] = 0
    dataset[
        f"project_reopen_rate_matured_last_{window_days}d"
    ] = np.nan

    TEMPORAL_INTEGER_FEATURE_COLUMNS.extend(
        [matured_col, reopened_col]
    )

for column_name in [
    "project_previous_matured_bugs",
    "project_previous_reopened_within_180d",
]:
    dataset[column_name] = 0
    TEMPORAL_INTEGER_FEATURE_COLUMNS.append(column_name)

dataset["project_previous_reopen_rate_180d"] = np.nan

project_sort_columns = [
    "Project_ID",
    "First_Resolution_Date",
    "First_Resolution_Change_ID",
    "Issue_ID",
]

ordered_for_project_history = dataset.sort_values(
    project_sort_columns,
    kind="mergesort",
)

for project_id, project_frame in (
    ordered_for_project_history.groupby(
        "Project_ID",
        sort=False,
    )
):
    project_index = project_frame.index

    resolution_dates = project_frame[
        "First_Resolution_Date"
    ].to_numpy(dtype="datetime64[ns]")

    project_time_indexed = project_frame.set_index(
        "First_Resolution_Date"
    )

    one_per_bug = pd.Series(
        1.0,
        index=project_time_indexed.index,
    )

    # Bugs resueltos recientemente. Se excluyen todos los bugs
    # resueltos exactamente en el mismo instante que el bug actual.
    for window_days in PROJECT_ACTIVITY_WINDOWS_DAYS:
        recent_resolved = (
            one_per_bug
            .rolling(
                f"{window_days}D",
                closed="left",
            )
            .sum()
            .fillna(0)
            .astype("int64")
            .to_numpy()
        )

        dataset.loc[
            project_index,
            f"project_resolved_last_{window_days}d",
        ] = recent_resolved

    # Medianas de atributos de bugs previamente resueltos.
    for window_days in PROJECT_CONTEXT_WINDOWS_DAYS:
        for source_col, output_suffix in (
            project_context_metric_map.items()
        ):
            rolling_median = (
                project_time_indexed[source_col]
                .rolling(
                    f"{window_days}D",
                    closed="left",
                    min_periods=1,
                )
                .median()
                .to_numpy()
            )

            dataset.loc[
                project_index,
                (
                    f"project_median_{output_suffix}_"
                    f"last_{window_days}d"
                ),
            ] = rolling_median

    # Cantidad de eventos de reapertura realmente observados antes
    # del instante de predicción.
    reopen_event_dates = np.sort(
        project_frame["First_Reopen_Date"]
        .dropna()
        .to_numpy(dtype="datetime64[ns]")
    )

    for window_days in PROJECT_ACTIVITY_WINDOWS_DAYS:
        if len(reopen_event_dates) == 0:
            recent_reopen_events = np.zeros(
                len(project_frame),
                dtype="int64",
            )
        else:
            left_positions = np.searchsorted(
                reopen_event_dates,
                resolution_dates
                - np.timedelta64(window_days, "D"),
                side="left",
            )
            right_positions = np.searchsorted(
                reopen_event_dates,
                resolution_dates,
                side="left",
            )
            recent_reopen_events = (
                right_positions - left_positions
            ).astype("int64")

        dataset.loc[
            project_index,
            f"project_reopen_events_last_{window_days}d",
        ] = recent_reopen_events

    # Resultados de reapertura a 180 días que ya estaban maduros
    # al resolver el bug actual.
    maturity_dates = project_frame[
        "_target_h180_maturity_date"
    ].to_numpy(dtype="datetime64[ns]")

    historical_targets = project_frame[
        "_target_h180_for_history"
    ].to_numpy(dtype="int64")

    cumulative_positives = np.concatenate(
        [
            np.asarray([0], dtype="int64"),
            np.cumsum(
                historical_targets,
                dtype="int64",
            ),
        ]
    )

    all_history_right = np.searchsorted(
        maturity_dates,
        resolution_dates,
        side="right",
    )

    all_history_n = all_history_right.astype("int64")
    all_history_positives = cumulative_positives[
        all_history_right
    ]

    dataset.loc[
        project_index,
        "project_previous_matured_bugs",
    ] = all_history_n

    dataset.loc[
        project_index,
        "project_previous_reopened_within_180d",
    ] = all_history_positives

    all_history_rate = np.divide(
        all_history_positives,
        all_history_n,
        out=np.full(
            len(project_frame),
            np.nan,
            dtype="float64",
        ),
        where=all_history_n > 0,
    )

    dataset.loc[
        project_index,
        "project_previous_reopen_rate_180d",
    ] = all_history_rate

    # Ventanas recientes definidas por la fecha en que el resultado
    # quedó completamente observable, no por la fecha de resolución.
    for window_days in PROJECT_CONTEXT_WINDOWS_DAYS:
        left_positions = np.searchsorted(
            maturity_dates,
            resolution_dates
            - np.timedelta64(window_days, "D"),
            side="left",
        )
        right_positions = np.searchsorted(
            maturity_dates,
            resolution_dates,
            side="right",
        )

        matured_n = (
            right_positions - left_positions
        ).astype("int64")

        matured_positives = (
            cumulative_positives[right_positions]
            - cumulative_positives[left_positions]
        )

        dataset.loc[
            project_index,
            f"project_matured_bugs_last_{window_days}d",
        ] = matured_n

        dataset.loc[
            project_index,
            (
                "project_reopened_within_180d_"
                f"matured_last_{window_days}d"
            ),
        ] = matured_positives

        matured_rate = np.divide(
            matured_positives,
            matured_n,
            out=np.full(
                len(project_frame),
                np.nan,
                dtype="float64",
            ),
            where=matured_n > 0,
        )

        dataset.loc[
            project_index,
            f"project_reopen_rate_matured_last_{window_days}d",
        ] = matured_rate


# ------------------------------------------------------------
# 2. Historia madura de los usuarios dentro del proyecto
# ------------------------------------------------------------

role_history_config = [
    (
        "Creator_ID_At_First_Resolution",
        "creator",
    ),
    (
        "Reporter_ID_At_First_Resolution",
        "reporter",
    ),
    (
        "Assignee_ID_At_First_Resolution",
        "assignee",
    ),
    (
        "Resolver_ID_At_First_Resolution",
        "resolver",
    ),
]


def add_matured_role_history(
    df,
    id_column,
    prefix,
):
    count_col = (
        f"{prefix}_previous_matured_bugs_in_project"
    )
    positive_col = (
        f"{prefix}_previous_reopened_within_180d_in_project"
    )
    rate_col = (
        f"{prefix}_previous_reopen_rate_180d_in_project"
    )
    smoothed_rate_col = (
        f"{prefix}_previous_reopen_rate_180d_"
        "smoothed_in_project"
    )

    df[count_col] = 0
    df[positive_col] = 0
    df[rate_col] = np.nan
    df[smoothed_rate_col] = np.nan

    TEMPORAL_INTEGER_FEATURE_COLUMNS.extend(
        [count_col, positive_col]
    )

    valid_mask = (
        df[id_column].notna()
        & df["First_Resolution_Date"].notna()
    )

    if not valid_mask.any():
        return

    queries = df.loc[
        valid_mask,
        [
            "Project_ID",
            id_column,
            "First_Resolution_Date",
        ],
    ].copy()

    queries["_row_index"] = queries.index
    queries["_project_key"] = (
        queries["Project_ID"].astype("string")
    )
    queries["_role_key"] = (
        queries[id_column].astype("string")
    )

    events = df.loc[
        valid_mask,
        [
            "Project_ID",
            id_column,
            "First_Resolution_Change_ID",
            "Issue_ID",
            "_target_h180_maturity_date",
            "_target_h180_for_history",
        ],
    ].copy()

    events["_project_key"] = (
        events["Project_ID"].astype("string")
    )
    events["_role_key"] = (
        events[id_column].astype("string")
    )

    events = events.sort_values(
        [
            "_target_h180_maturity_date",
            "_project_key",
            "_role_key",
            "First_Resolution_Change_ID",
            "Issue_ID",
        ],
        kind="mergesort",
    )

    events["_matured_count"] = (
        events
        .groupby(
            ["_project_key", "_role_key"],
            sort=False,
        )
        .cumcount()
        .add(1)
        .astype("int64")
    )

    events["_matured_positives"] = (
        events
        .groupby(
            ["_project_key", "_role_key"],
            sort=False,
        )["_target_h180_for_history"]
        .cumsum()
        .astype("int64")
    )

    queries = queries.sort_values(
        [
            "First_Resolution_Date",
            "_project_key",
            "_role_key",
            "_row_index",
        ],
        kind="mergesort",
    )

    matched = pd.merge_asof(
        queries,
        events[
            [
                "_project_key",
                "_role_key",
                "_target_h180_maturity_date",
                "_matured_count",
                "_matured_positives",
            ]
        ],
        left_on="First_Resolution_Date",
        right_on="_target_h180_maturity_date",
        by=["_project_key", "_role_key"],
        direction="backward",
        allow_exact_matches=True,
    )

    matured_count = (
        matched["_matured_count"]
        .fillna(0)
        .astype("int64")
        .to_numpy()
    )

    matured_positives = (
        matched["_matured_positives"]
        .fillna(0)
        .astype("int64")
        .to_numpy()
    )

    raw_rate = np.divide(
        matured_positives,
        matured_count,
        out=np.full(
            len(matched),
            np.nan,
            dtype="float64",
        ),
        where=matured_count > 0,
    )

    matched_index = matched["_row_index"].to_numpy()

    project_prior_rate = df.loc[
        matched_index,
        "project_previous_reopen_rate_180d",
    ].to_numpy(dtype="float64")

    smoothed_rate = np.divide(
        matured_positives
        + USER_RATE_SMOOTHING_STRENGTH
        * project_prior_rate,
        matured_count
        + USER_RATE_SMOOTHING_STRENGTH,
        out=np.full(
            len(matched),
            np.nan,
            dtype="float64",
        ),
        where=np.isfinite(project_prior_rate),
    )

    df.loc[matched_index, count_col] = matured_count
    df.loc[matched_index, positive_col] = matured_positives
    df.loc[matched_index, rate_col] = raw_rate
    df.loc[matched_index, smoothed_rate_col] = smoothed_rate


for id_column, prefix in role_history_config:
    add_matured_role_history(
        dataset,
        id_column=id_column,
        prefix=prefix,
    )


# ------------------------------------------------------------
# 3. Variables relativas al contexto reciente del proyecto
# ------------------------------------------------------------


def neutral_ratio(numerator, denominator, neutral_value=1.0):
    """Divide y usa un valor neutral únicamente para el caso 0/0.

    Los faltantes reales se conservan para ser tratados dentro de cada
    fold de entrenamiento, sin utilizar información futura.
    """
    numerator_numeric = pd.to_numeric(
        numerator,
        errors="coerce",
    )
    denominator_numeric = pd.to_numeric(
        denominator,
        errors="coerce",
    )

    result = pd.Series(
        np.nan,
        index=numerator_numeric.index,
        dtype="float64",
    )

    valid_nonzero = (
        numerator_numeric.notna()
        & denominator_numeric.notna()
        & denominator_numeric.ne(0)
    )
    result.loc[valid_nonzero] = (
        numerator_numeric.loc[valid_nonzero]
        / denominator_numeric.loc[valid_nonzero]
    )

    both_zero = (
        numerator_numeric.eq(0)
        & denominator_numeric.eq(0)
    )
    result.loc[both_zero] = float(neutral_value)

    return result


def pseudocount_ratio(numerator, denominator, pseudocount=1.0):
    """Compara magnitudes no negativas evitando divisiones por cero.

    (actual + 1) / (referencia + 1) conserva 1 como valor de igualdad
    y permite comparar variables de conteo cuya mediana puede ser cero.
    """
    numerator_numeric = pd.to_numeric(
        numerator,
        errors="coerce",
    )
    denominator_numeric = pd.to_numeric(
        denominator,
        errors="coerce",
    )

    valid = (
        numerator_numeric.notna()
        & denominator_numeric.notna()
    )

    result = pd.Series(
        np.nan,
        index=numerator_numeric.index,
        dtype="float64",
    )
    result.loc[valid] = (
        numerator_numeric.loc[valid] + pseudocount
    ) / (
        denominator_numeric.loc[valid] + pseudocount
    )

    return result


ratio_source_config = {
    "resolution_time_hours": "resolution_time",
    "n_comments_until_resolution": "comments",
    "n_status_events_until_resolution": "status_events",
    "n_changes_until_resolution": "changes",
}

for window_days in PROJECT_CONTEXT_WINDOWS_DAYS:
    for source_col, short_name in ratio_source_config.items():
        median_col = (
            "project_median_"
            f"{project_context_metric_map[source_col]}_"
            f"last_{window_days}d"
        )

        ratio_col = (
            f"{short_name}_vs_project_median_{window_days}d"
        )

        # La mediana puede ser cero (por ejemplo, cero comentarios).
        # El pseudoconteo evita infinitos y conserva 1 como igualdad.
        dataset[ratio_col] = (
            pseudocount_ratio(
                dataset[source_col],
                dataset[median_col],
                pseudocount=1.0,
            )
            .fillna(1.0)
        )


# Las columnas auxiliares contienen resultado futuro y deben eliminarse.
dataset.drop(
    columns=[
        "_target_h180_for_history",
        "_target_h180_maturity_date",
    ],
    inplace=True,
)

TEMPORAL_ENGINEERED_FEATURE_COLUMNS = [
    col
    for col in dataset.columns
    if col not in columns_before_temporal_engineering
]

print(
    "Variables temporales e históricas agregadas:",
    len(TEMPORAL_ENGINEERED_FEATURE_COLUMNS),
)

assert not {
    "_target_h180_for_history",
    "_target_h180_maturity_date",
}.intersection(dataset.columns)

# Auditoría de faltantes esperables. No se imputan.
temporal_feature_audit = pd.DataFrame({
    "variable": TEMPORAL_ENGINEERED_FEATURE_COLUMNS,
    "dtype": [
        str(dataset[col].dtype)
        for col in TEMPORAL_ENGINEERED_FEATURE_COLUMNS
    ],
    "missing": [
        int(dataset[col].isna().sum())
        for col in TEMPORAL_ENGINEERED_FEATURE_COLUMNS
    ],
    "n_unique": [
        int(dataset[col].nunique(dropna=True))
        for col in TEMPORAL_ENGINEERED_FEATURE_COLUMNS
    ],
})

display(temporal_feature_audit)


/tmp/ipykernel_37/2496552759.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset["_target_h180_maturity_date"] = (
/tmp/ipykernel_37/2496552759.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset[resolved_col] = 0
/tmp/ipykernel_37/2496552759.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fra

Variables temporales e históricas agregadas: 47


/tmp/ipykernel_37/2496552759.py:649: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset[ratio_col] = (
/tmp/ipykernel_37/2496552759.py:649: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset[ratio_col] = (
/tmp/ipykernel_37/2496552759.py:649: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  datase

,variable,dtype,missing,n_unique
0,project_resolved_last_30d,int64,0,2243
1,project_reopen_events_last_30d,int64,0,195
2,project_resolved_last_90d,int64,0,2584
3,project_reopen_events_last_90d,int64,0,359
4,project_resolved_last_180d,int64,0,3108
5,project_reopen_events_last_180d,int64,0,544
6,project_median_resolution_time_hours_last_90d,float64,20,19895
7,project_median_comments_until_resolution_last_90d,float64,20,27
8,project_median_status_events_until_resolution_...,float64,20,17
9,project_median_changes_until_resolution_last_90d,float64,20,98


### Tendencias, deltas y contexto histórico adicional

Este bloque agrega señales de cambio temporal sin utilizar información posterior al momento de la primera resolución. Los cocientes se construyen con valores neutrales cuando la operación no está definida por ausencia total de historia: un ratio neutral vale 1 y una diferencia o tendencia neutral vale 0. Los conteos históricos asociados permanecen en el dataset y permiten distinguir entre un valor realmente observado y la ausencia de antecedentes.


In [82]:
# ============================================================
# VARIABLES AVANZADAS DE TENDENCIA Y CAMBIO HISTÓRICO
# ============================================================

ADVANCED_TREND_BIN_DAYS = 90
ADVANCED_TREND_N_BINS = 4
ADVANCED_TREND_MIN_MATURED_PER_BIN = 3
USER_RECENT_HISTORY_WINDOW_DAYS = 180

columns_before_advanced_temporal_engineering = set(dataset.columns)


# ------------------------------------------------------------
# 1. Deltas y ratios entre ventanas recientes del proyecto
# ------------------------------------------------------------

project_previous_90d_matured = (
    dataset["project_matured_bugs_last_180d"]
    - dataset["project_matured_bugs_last_90d"]
).clip(lower=0)

project_previous_90d_reopened = (
    dataset[
        "project_reopened_within_180d_matured_last_180d"
    ]
    - dataset[
        "project_reopened_within_180d_matured_last_90d"
    ]
).clip(lower=0)

dataset["project_matured_bugs_previous_90d"] = (
    project_previous_90d_matured.astype("int64")
)
dataset[
    "project_reopened_within_180d_matured_previous_90d"
] = project_previous_90d_reopened.astype("int64")

TEMPORAL_INTEGER_FEATURE_COLUMNS.extend([
    "project_matured_bugs_previous_90d",
    "project_reopened_within_180d_matured_previous_90d",
])

dataset["project_reopen_rate_matured_previous_90d"] = np.divide(
    project_previous_90d_reopened.to_numpy(dtype="float64"),
    project_previous_90d_matured.to_numpy(dtype="float64"),
    out=np.full(len(dataset), np.nan, dtype="float64"),
    where=project_previous_90d_matured.to_numpy(dtype="float64") > 0,
)

dataset[
    "project_reopen_rate_change_recent_vs_previous_90d"
] = (
    dataset["project_reopen_rate_matured_last_90d"]
    - dataset["project_reopen_rate_matured_previous_90d"]
).fillna(0.0)

dataset["project_reopen_rate_ratio_90d_vs_180d"] = (
    neutral_ratio(
        dataset["project_reopen_rate_matured_last_90d"],
        dataset["project_reopen_rate_matured_last_180d"],
        neutral_value=1.0,
    )
    .fillna(1.0)
)

dataset[
    "project_resolution_activity_ratio_30d_vs_180d_average"
] = (
    neutral_ratio(
        dataset["project_resolved_last_30d"],
        dataset["project_resolved_last_180d"] / 6.0,
        neutral_value=1.0,
    )
    .fillna(1.0)
)

dataset[
    "project_reopen_event_activity_ratio_30d_vs_180d_average"
] = (
    neutral_ratio(
        dataset["project_reopen_events_last_30d"],
        dataset["project_reopen_events_last_180d"] / 6.0,
        neutral_value=1.0,
    )
    .fillna(1.0)
)

for source_col, output_suffix in project_context_metric_map.items():
    median_90_col = (
        f"project_median_{output_suffix}_last_90d"
    )
    median_180_col = (
        f"project_median_{output_suffix}_last_180d"
    )

    dataset[
        f"project_median_{output_suffix}_change_90d_vs_180d"
    ] = (
        dataset[median_90_col]
        - dataset[median_180_col]
    ).fillna(0.0)

    dataset[
        f"project_median_{output_suffix}_ratio_90d_vs_180d"
    ] = (
        pseudocount_ratio(
            dataset[median_90_col],
            dataset[median_180_col],
            pseudocount=1.0,
        )
        .fillna(1.0)
    )


# ------------------------------------------------------------
# 2. Tendencia de la tasa madura en cuatro trimestres
# ------------------------------------------------------------

advanced_days_to_first_reopen = (
    dataset["First_Reopen_Date"]
    - dataset["First_Resolution_Date"]
).dt.total_seconds() / 86400

advanced_target_h180 = (
    advanced_days_to_first_reopen
    .between(
        0,
        TEMPORAL_HISTORY_HORIZON_DAYS,
        inclusive="both",
    )
    .fillna(False)
    .astype("int8")
)

advanced_maturity_date = (
    dataset["First_Resolution_Date"]
    + pd.Timedelta(days=TEMPORAL_HISTORY_HORIZON_DAYS)
)

dataset["project_reopen_rate_trend_4q"] = np.nan
dataset["project_reopen_rate_volatility_4q"] = np.nan
dataset["project_reopen_rate_change_oldest_to_recent_4q"] = np.nan
dataset["project_reopen_rate_quarters_available_4q"] = 0

TEMPORAL_INTEGER_FEATURE_COLUMNS.append(
    "project_reopen_rate_quarters_available_4q"
)

ordered_for_advanced_project_history = (
    dataset
    .assign(
        _advanced_target_h180=advanced_target_h180,
        _advanced_maturity_date=advanced_maturity_date,
    )
    .sort_values(
        project_sort_columns,
        kind="mergesort",
    )
)

for project_id, project_frame in (
    ordered_for_advanced_project_history.groupby(
        "Project_ID",
        sort=False,
    )
):
    project_index = project_frame.index
    n_project_rows = len(project_frame)

    resolution_dates = project_frame[
        "First_Resolution_Date"
    ].to_numpy(dtype="datetime64[ns]")

    maturity_dates = project_frame[
        "_advanced_maturity_date"
    ].to_numpy(dtype="datetime64[ns]")

    target_values = project_frame[
        "_advanced_target_h180"
    ].to_numpy(dtype="int64")

    cumulative_positives = np.concatenate([
        np.asarray([0], dtype="int64"),
        np.cumsum(target_values, dtype="int64"),
    ])

    rates_recent_to_old = np.full(
        (n_project_rows, ADVANCED_TREND_N_BINS),
        np.nan,
        dtype="float64",
    )

    for bin_number in range(ADVANCED_TREND_N_BINS):
        upper_dates = (
            resolution_dates
            - np.timedelta64(
                bin_number * ADVANCED_TREND_BIN_DAYS,
                "D",
            )
        )
        lower_dates = (
            resolution_dates
            - np.timedelta64(
                (bin_number + 1) * ADVANCED_TREND_BIN_DAYS,
                "D",
            )
        )

        upper_positions = np.searchsorted(
            maturity_dates,
            upper_dates,
            side="right",
        )
        lower_positions = np.searchsorted(
            maturity_dates,
            lower_dates,
            side="right",
        )

        bin_counts = (
            upper_positions - lower_positions
        ).astype("int64")

        bin_positives = (
            cumulative_positives[upper_positions]
            - cumulative_positives[lower_positions]
        )

        rates_recent_to_old[:, bin_number] = np.divide(
            bin_positives,
            bin_counts,
            out=np.full(
                n_project_rows,
                np.nan,
                dtype="float64",
            ),
            where=(
                bin_counts
                >= ADVANCED_TREND_MIN_MATURED_PER_BIN
            ),
        )

    # Para estimar la pendiente, x avanza desde el trimestre más
    # antiguo hacia el más reciente.
    rates_old_to_recent = rates_recent_to_old[:, ::-1]
    x_values = np.arange(
        ADVANCED_TREND_N_BINS,
        dtype="float64",
    )

    finite_mask = np.isfinite(rates_old_to_recent)
    n_available = finite_mask.sum(axis=1).astype("int64")

    x_matrix = np.broadcast_to(
        x_values,
        rates_old_to_recent.shape,
    )

    safe_rates = np.where(
        finite_mask,
        rates_old_to_recent,
        0.0,
    )
    safe_x = np.where(
        finite_mask,
        x_matrix,
        0.0,
    )

    sum_x = safe_x.sum(axis=1)
    sum_y = safe_rates.sum(axis=1)
    sum_xx = (safe_x * safe_x).sum(axis=1)
    sum_xy = (safe_x * safe_rates).sum(axis=1)

    denominator = (
        n_available * sum_xx
        - sum_x * sum_x
    )

    slope = np.divide(
        n_available * sum_xy - sum_x * sum_y,
        denominator,
        out=np.full(
            n_project_rows,
            np.nan,
            dtype="float64",
        ),
        where=(
            (n_available >= 2)
            & (denominator > 0)
        ),
    )

    mean_rate = np.divide(
        sum_y,
        n_available,
        out=np.full(
            n_project_rows,
            np.nan,
            dtype="float64",
        ),
        where=n_available > 0,
    )

    centered = np.where(
        finite_mask,
        rates_old_to_recent - mean_rate[:, None],
        0.0,
    )

    volatility = np.sqrt(
        np.divide(
            (centered * centered).sum(axis=1),
            n_available,
            out=np.full(
                n_project_rows,
                np.nan,
                dtype="float64",
            ),
            where=n_available >= 2,
        )
    )

    oldest_rate = rates_old_to_recent[:, 0]
    recent_rate = rates_old_to_recent[:, -1]

    oldest_to_recent_change = np.where(
        np.isfinite(oldest_rate)
        & np.isfinite(recent_rate),
        recent_rate - oldest_rate,
        np.nan,
    )

    dataset.loc[
        project_index,
        "project_reopen_rate_trend_4q",
    ] = slope

    dataset.loc[
        project_index,
        "project_reopen_rate_volatility_4q",
    ] = volatility

    dataset.loc[
        project_index,
        "project_reopen_rate_change_oldest_to_recent_4q",
    ] = oldest_to_recent_change

    dataset.loc[
        project_index,
        "project_reopen_rate_quarters_available_4q",
    ] = n_available


# Cuando no hay al menos dos trimestres utilizables, la tendencia,
# la volatilidad y el cambio no son estimables. Se usa 0 como valor
# neutral; la cantidad de trimestres disponibles conserva la señal
# de que no existía historia suficiente.
for neutral_trend_col in [
    "project_reopen_rate_trend_4q",
    "project_reopen_rate_volatility_4q",
    "project_reopen_rate_change_oldest_to_recent_4q",
]:
    dataset[neutral_trend_col] = (
        dataset[neutral_trend_col].fillna(0.0)
    )


# ------------------------------------------------------------
# 3. Historia reciente de cada usuario frente a su trayectoria
# ------------------------------------------------------------

def normalize_history_key(value):
    if pd.isna(value):
        return None
    return str(value)


def add_recent_role_history_memory_safe(
    df,
    id_column,
    prefix,
):
    recent_count_col = (
        f"{prefix}_recent_matured_bugs_last_180d_in_project"
    )
    recent_positive_col = (
        f"{prefix}_recent_reopened_within_180d_"
        "last_180d_in_project"
    )
    recent_rate_col = (
        f"{prefix}_recent_reopen_rate_180d_in_project"
    )
    recent_change_col = (
        f"{prefix}_recent_vs_lifetime_reopen_rate_"
        "change_in_project"
    )

    df[recent_count_col] = 0
    df[recent_positive_col] = 0
    df[recent_rate_col] = np.nan
    df[recent_change_col] = np.nan

    TEMPORAL_INTEGER_FEATURE_COLUMNS.extend([
        recent_count_col,
        recent_positive_col,
    ])

    ordered = (
        df
        .sort_values(
            project_sort_columns,
            kind="mergesort",
        )
    )

    for project_id, project_frame in ordered.groupby(
        "Project_ID",
        sort=False,
    ):
        project_index = project_frame.index
        n_rows = len(project_frame)

        resolution_dates = project_frame[
            "First_Resolution_Date"
        ].to_numpy(dtype="datetime64[ns]")

        maturity_dates = (
            project_frame["First_Resolution_Date"]
            + pd.Timedelta(
                days=TEMPORAL_HISTORY_HORIZON_DAYS
            )
        ).to_numpy(dtype="datetime64[ns]")

        targets = (
            (
                project_frame["First_Reopen_Date"]
                - project_frame["First_Resolution_Date"]
            )
            .dt.total_seconds()
            .div(86400)
            .between(
                0,
                TEMPORAL_HISTORY_HORIZON_DAYS,
                inclusive="both",
            )
            .fillna(False)
            .astype("int8")
            .to_numpy(dtype="int8")
        )

        role_values = project_frame[
            id_column
        ].to_numpy(dtype="object")

        recent_counts = np.zeros(
            n_rows,
            dtype="int64",
        )
        recent_positives = np.zeros(
            n_rows,
            dtype="int64",
        )
        recent_rates = np.full(
            n_rows,
            np.nan,
            dtype="float64",
        )

        event_queues = defaultdict(deque)
        event_positive_counts = defaultdict(int)
        event_pointer = 0

        for row_position, current_date in enumerate(
            resolution_dates
        ):
            while (
                event_pointer < n_rows
                and maturity_dates[event_pointer]
                <= current_date
            ):
                event_user = normalize_history_key(
                    role_values[event_pointer]
                )

                if event_user is not None:
                    event_target = int(
                        targets[event_pointer]
                    )
                    event_queues[event_user].append(
                        (
                            maturity_dates[event_pointer],
                            event_target,
                        )
                    )
                    event_positive_counts[
                        event_user
                    ] += event_target

                event_pointer += 1

            current_user = normalize_history_key(
                role_values[row_position]
            )

            if current_user is None:
                continue

            lower_date = (
                current_date
                - np.timedelta64(
                    USER_RECENT_HISTORY_WINDOW_DAYS,
                    "D",
                )
            )

            current_queue = event_queues[current_user]

            while (
                current_queue
                and current_queue[0][0] < lower_date
            ):
                _, expired_target = current_queue.popleft()
                event_positive_counts[
                    current_user
                ] -= int(expired_target)

            current_count = len(current_queue)
            current_positives = int(
                event_positive_counts[current_user]
            )

            recent_counts[row_position] = current_count
            recent_positives[
                row_position
            ] = current_positives

            if current_count > 0:
                recent_rates[row_position] = (
                    current_positives
                    / current_count
                )

        lifetime_rate_col = (
            f"{prefix}_previous_reopen_rate_180d_in_project"
        )

        lifetime_rates = project_frame[
            lifetime_rate_col
        ].to_numpy(dtype="float64")

        recent_changes = np.where(
            np.isfinite(recent_rates)
            & np.isfinite(lifetime_rates),
            recent_rates - lifetime_rates,
            0.0,
        )

        df.loc[
            project_index,
            recent_count_col,
        ] = recent_counts

        df.loc[
            project_index,
            recent_positive_col,
        ] = recent_positives

        df.loc[
            project_index,
            recent_rate_col,
        ] = recent_rates

        df.loc[
            project_index,
            recent_change_col,
        ] = recent_changes


for id_column, prefix in role_history_config:
    add_recent_role_history_memory_safe(
        dataset,
        id_column=id_column,
        prefix=prefix,
    )


ADVANCED_TEMPORAL_FEATURE_COLUMNS = [
    col
    for col in dataset.columns
    if col not in columns_before_advanced_temporal_engineering
]

print(
    "Variables avanzadas de tendencia agregadas:",
    len(ADVANCED_TEMPORAL_FEATURE_COLUMNS),
)

advanced_temporal_feature_audit = pd.DataFrame({
    "variable": ADVANCED_TEMPORAL_FEATURE_COLUMNS,
    "dtype": [
        str(dataset[col].dtype)
        for col in ADVANCED_TEMPORAL_FEATURE_COLUMNS
    ],
    "missing": [
        int(dataset[col].isna().sum())
        for col in ADVANCED_TEMPORAL_FEATURE_COLUMNS
    ],
    "n_unique": [
        int(dataset[col].nunique(dropna=True))
        for col in ADVANCED_TEMPORAL_FEATURE_COLUMNS
    ],
})

display(advanced_temporal_feature_audit)


/tmp/ipykernel_37/143493822.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset["project_matured_bugs_previous_90d"] = (
/tmp/ipykernel_37/143493822.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset[
/tmp/ipykernel_37/143493822.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  

Variables avanzadas de tendencia agregadas: 35


,variable,dtype,missing,n_unique
0,project_matured_bugs_previous_90d,int64,0,1496
1,project_reopened_within_180d_matured_previous_90d,int64,0,321
2,project_reopen_rate_matured_previous_90d,float64,10864,9026
3,project_reopen_rate_change_recent_vs_previous_90d,float64,0,20172
4,project_reopen_rate_ratio_90d_vs_180d,float64,0,19908
5,project_resolution_activity_ratio_30d_vs_180d_...,float64,0,43536
6,project_reopen_event_activity_ratio_30d_vs_180...,float64,0,3583
7,project_median_resolution_time_hours_change_90...,float64,0,45601
8,project_median_resolution_time_hours_ratio_90d...,float64,0,45733
9,project_median_comments_until_resolution_chang...,float64,0,17


## 17. Controles finales del dataset

Se controla que exista una sola fila por bug, que los eventos usados como predictores sean observables al momento de la primera resolución y que los valores neutrales introducidos en diferencias, tendencias y ratios no generen infinitos. Las tasas y medianas históricas que dependen de antecedentes reales pueden continuar como `NaN`; se completan dentro de cada fold del notebook de modelado para evitar fuga temporal.


In [83]:
dataset.duplicated(["Issue_ID", "Project_ID"]).sum()

np.int64(0)

In [84]:
final_target_summary = pd.DataFrame({
    "metric": [
        "filas_dataset",
        "bugs_reabiertos",
        "bugs_no_reabiertos",
        "tasa_reapertura_pct",
        "variables_totales"
    ],
    "value": [
        len(dataset),
        int(dataset["Reopened_Target"].sum()),
        int((dataset["Reopened_Target"] == 0).sum()),
        round(100 * dataset["Reopened_Target"].mean(), 2),
        dataset.shape[1]
    ]
})

final_target_summary

,metric,value
0,filas_dataset,67806.00
1,bugs_reabiertos,7635.00
2,bugs_no_reabiertos,60171.00
3,tasa_reapertura_pct,11.26
4,variables_totales,198.00


In [85]:
missing_summary = (
    dataset
    .isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

missing_summary.columns = ["variable", "missing_count"]
missing_summary["missing_pct"] = (
    100 * missing_summary["missing_count"] / len(dataset)
).round(2)

missing_summary.head(30)


,variable,missing_count,missing_pct
0,Story_Point_At_First_Resolution,60447,89.15
1,First_Reopen_Group,60171,88.74
2,First_Reopen_Status,60171,88.74
3,First_Reopen_Date,60171,88.74
4,First_Reopen_Change_ID,60171,88.74
5,creator_recent_reopen_rate_180d_in_project,32746,48.29
6,reporter_recent_reopen_rate_180d_in_project,32726,48.26
7,assignee_recent_reopen_rate_180d_in_project,27075,39.93
8,creator_previous_reopen_rate_180d_in_project,25771,38.01
9,reporter_previous_reopen_rate_180d_in_project,25749,37.97


In [86]:
assert dataset.duplicated(KEY_COLS).sum() == 0

experience_count_cols = [
    "creator_previous_resolved_bugs_in_project",
    "reporter_previous_resolved_bugs_in_project",
    "assignee_previous_resolved_bugs_in_project",
    "resolver_previous_resolutions_in_project"
]


assert (
    dataset[experience_count_cols] >= 0
).all().all()


assert set(
    dataset[
        [
            "creator_is_new_in_project",
            "reporter_is_new_in_project",
            "assignee_is_new_in_project",
            "resolver_is_new_in_project"
        ]
    ]
    .stack()
    .unique()
).issubset({0, 1})


assert changes_until_resolution_temporal_ok

assert comments_until_resolution_temporal_ok

reopened_rows = bug_target["Reopened_Target"].eq(1)

assert (
    (
        bug_target.loc[reopened_rows, "First_Reopen_Date"]
        > bug_target.loc[reopened_rows, "First_Resolution_Date"]
    )
    |
    (
        (
            bug_target.loc[reopened_rows, "First_Reopen_Date"]
            == bug_target.loc[reopened_rows, "First_Resolution_Date"]
        )
        &
        (
            bug_target.loc[reopened_rows, "First_Reopen_Change_ID"]
            > bug_target.loc[reopened_rows, "First_Resolution_Change_ID"]
        )
    )
).all()

print("Controles temporales: OK")


Controles temporales: OK


## 18. Separación de variables para modelado

Se excluyen identificadores, textos crudos, fechas y toda información posterior a la
primera resolución. Los campos mutables finales de `issue` no forman parte del dataset.

In [87]:
nullable_integer_cols = USER_ID_COLUMNS


for col in nullable_integer_cols:
    if col in dataset.columns:
        dataset[col] = (
            pd.to_numeric(
                dataset[col],
                errors="coerce"
            )
            .astype("Int64")
        )


integer_feature_cols = [
    col
    for col in dataset.columns
    if (
        col.startswith("n_")
        or col.startswith("has_")
        or col.startswith("had_")
        or col in USER_RELATIONSHIP_COLUMNS
        or col in experience_cols
        or col in TEMPORAL_INTEGER_FEATURE_COLUMNS
        or col in [
            "title_length",
            "description_text_length",
            "description_code_length",
            "creation_year",
            "creation_month",
            "creation_weekday",
            "Reopened_Target"
        ]
    )
]


for col in integer_feature_cols:
    converted = pd.to_numeric(
        dataset[col],
        errors="raise"
    )

    if converted.isna().any():
        raise ValueError(
            f"La columna entera {col} "
            "contiene valores nulos inesperados"
        )

    dataset[col] = (
        converted
        .round()
        .astype("int64")
    )


dataset[
    nullable_integer_cols
    + [
        col
        for col in integer_feature_cols
        if col in dataset.columns
    ]
].dtypes

Creator_ID_At_First_Resolution                               Int64
Reporter_ID_At_First_Resolution                              Int64
Assignee_ID_At_First_Resolution                              Int64
Resolver_ID_At_First_Resolution                              Int64
Reopened_Target                                              int64
                                                             ...  
reporter_recent_reopened_within_180d_last_180d_in_project    int64
assignee_recent_matured_bugs_last_180d_in_project            int64
assignee_recent_reopened_within_180d_last_180d_in_project    int64
resolver_recent_matured_bugs_last_180d_in_project            int64
resolver_recent_reopened_within_180d_last_180d_in_project    int64
Length: 112, dtype: object

In [88]:
target_col = "Reopened_Target"


identifier_columns = [
    "Issue_ID",
    "Issue_Key",
    "First_Resolution_Change_ID"
]


raw_date_columns = [
    "Creation_Date",
    "First_Resolution_Date",
    "First_Reopen_Date",
    "First_Comment_Date",
    "Last_Comment_Date",
    "Project_Observation_End_Date",
]


future_outcome_columns = [
    "First_Reopen_Change_ID",
    "First_Reopen_Date",
    "First_Reopen_Status",
    "First_Reopen_Group"
]


constant_or_duplicate_columns = [
    "First_Resolution_Group",
    "n_change_priority_until_resolution",
    "n_change_status_until_resolution"
]


raw_text_columns = [
    "Title_At_First_Resolution",
    "Description_At_First_Resolution"
]

temporarily_excluded_structural_columns = [
    "n_issue_links_at_first_resolution",
    "has_issue_links_at_first_resolution"
]


In [89]:
model_source = dataset.copy()

model_source["Project_ID"] = (
    model_source["Project_ID"]
    .astype("string")
)

In [90]:
base_excluded_columns = (
    identifier_columns
    + raw_date_columns
    + future_outcome_columns
    + constant_or_duplicate_columns
    + raw_text_columns
    + temporarily_excluded_structural_columns
    + USER_ID_COLUMNS
)


base_modeling_features = [
    col
    for col in model_source.columns
    if col not in base_excluded_columns
    and col != target_col
]


modeling_dataset = model_source[
    base_modeling_features + [target_col]
].copy()

In [91]:
with_users_excluded_columns = (
    identifier_columns
    + raw_date_columns
    + future_outcome_columns
    + constant_or_duplicate_columns
    + raw_text_columns
    + temporarily_excluded_structural_columns
)


with_users_features = [
    col
    for col in model_source.columns
    if col not in with_users_excluded_columns
    and col != target_col
]


modeling_dataset_with_users = model_source[
    with_users_features + [target_col]
].copy()

In [92]:
for col in USER_ID_COLUMNS:
    modeling_dataset_with_users[col] = (
        modeling_dataset_with_users[col]
        .astype("string")
        .fillna("__MISSING__")
    )

In [93]:
text_dataset = dataset[
    [
        "Issue_ID",
        "Project_ID",
        "Issue_Key",
        "First_Resolution_Date",
        "Title_At_First_Resolution",
        "Description_At_First_Resolution",
        "Reopened_Target"
    ]
].copy()

In [94]:
forbidden_leakage_columns = {
    "First_Reopen_Change_ID",
    "First_Reopen_Date",
    "First_Reopen_Status",
    "First_Reopen_Group",

    # Valores finales mutables que no deben aparecer
    "Priority",
    "Assignee_ID",
    "Creator_ID",
    "Reporter_ID",
    "Story_Point",
    "Title",
    "Description",
    "Description_Text",
    "Description_Code",
    "Pull_Request_URL"
}


datasets_for_leakage_check = {
    "modeling_dataset": modeling_dataset,
    "modeling_dataset_with_users": (
        modeling_dataset_with_users
    )
}


for dataset_name, candidate_dataset in (
    datasets_for_leakage_check.items()
):
    leaked_columns = sorted(
        forbidden_leakage_columns.intersection(
            candidate_dataset.columns
        )
    )

    if leaked_columns:
        raise ValueError(
            f"{dataset_name} contiene columnas con fuga: "
            f"{leaked_columns}"
        )

In [95]:
unexpected_user_columns_in_base = sorted(
    set(USER_ID_COLUMNS).intersection(
        modeling_dataset.columns
    )
)

if unexpected_user_columns_in_base:
    raise ValueError(
        "El modelo base contiene IDs de usuarios: "
        f"{unexpected_user_columns_in_base}"
    )


missing_user_columns = sorted(
    set(USER_ID_COLUMNS).difference(
        modeling_dataset_with_users.columns
    )
)

if missing_user_columns:
    raise ValueError(
        "Faltan IDs en el dataset con usuarios: "
        f"{missing_user_columns}"
    )


print(
    "Dataset base:",
    modeling_dataset.shape
)

print(
    "Dataset con usuarios:",
    modeling_dataset_with_users.shape
)

print(
    "Dataset de texto:",
    text_dataset.shape
)

Dataset base: (67806, 175)
Dataset con usuarios: (67806, 179)
Dataset de texto: (67806, 7)


In [96]:
def inspect_modeling_columns(df, dataset_name, target_col):
    print(f"\n{dataset_name}")
    print("-" * len(dataset_name))

    print("\nTipos de datos:")
    display(df.dtypes.value_counts())

    categorical_cols = df.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()

    numeric_cols = df.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    numeric_cols = [
        col
        for col in numeric_cols
        if col != target_col
    ]

    print("\nVariables categóricas:")
    print(len(categorical_cols))
    print(categorical_cols)

    print("\nVariables numéricas:")
    print(len(numeric_cols))
    print(numeric_cols[:30])

    return categorical_cols, numeric_cols

In [97]:
base_categorical_cols, base_numeric_cols = (
    inspect_modeling_columns(
        modeling_dataset,
        dataset_name="Dataset base",
        target_col=target_col
    )
)


Dataset base
------------

Tipos de datos:


int64      104
float64     65
str          3
string       2
Float64      1
Name: count, dtype: int64


Variables categóricas:
5
['Project_ID', 'Status_Before_First_Resolution', 'Status_Group_Before_First_Resolution', 'First_Resolution_Status', 'Priority_At_First_Resolution']

Variables numéricas:
169
['resolution_time_hours', 'resolution_time_days', 'creation_year', 'creation_month', 'creation_weekday', 'resolution_year', 'resolution_month', 'resolution_quarter', 'resolution_weekday', 'resolution_month_sin', 'resolution_month_cos', 'days_since_project_start', 'project_age_years', 'Story_Point_At_First_Resolution', 'title_length', 'has_title', 'description_text_length', 'has_description_text', 'description_code_length', 'has_description_code', 'has_pull_request', 'has_creator', 'has_reporter', 'has_assignee', 'has_resolver', 'creator_eq_reporter', 'creator_eq_assignee', 'reporter_eq_assignee', 'resolver_eq_assignee', 'resolver_eq_reporter']


In [98]:
users_categorical_cols, users_numeric_cols = (
    inspect_modeling_columns(
        modeling_dataset_with_users,
        dataset_name="Dataset con usuarios",
        target_col=target_col
    )
)


Dataset con usuarios
--------------------

Tipos de datos:


int64      104
float64     65
string       6
str          3
Float64      1
Name: count, dtype: int64


Variables categóricas:
9
['Project_ID', 'Resolver_ID_At_First_Resolution', 'Status_Before_First_Resolution', 'Status_Group_Before_First_Resolution', 'First_Resolution_Status', 'Priority_At_First_Resolution', 'Creator_ID_At_First_Resolution', 'Reporter_ID_At_First_Resolution', 'Assignee_ID_At_First_Resolution']

Variables numéricas:
169
['resolution_time_hours', 'resolution_time_days', 'creation_year', 'creation_month', 'creation_weekday', 'resolution_year', 'resolution_month', 'resolution_quarter', 'resolution_weekday', 'resolution_month_sin', 'resolution_month_cos', 'days_since_project_start', 'project_age_years', 'Story_Point_At_First_Resolution', 'title_length', 'has_title', 'description_text_length', 'has_description_text', 'description_code_length', 'has_description_code', 'has_pull_request', 'has_creator', 'has_reporter', 'has_assignee', 'has_resolver', 'creator_eq_reporter', 'creator_eq_assignee', 'reporter_eq_assignee', 'resolver_eq_assignee', 'resolver_eq_reporter']


In [99]:
assert not set(USER_ID_COLUMNS).intersection(
    modeling_dataset.columns
), (
    "El dataset base contiene IDs de usuarios"
)


assert set(USER_ID_COLUMNS).issubset(
    users_categorical_cols
), (
    "No todos los IDs de usuarios fueron tratados "
    "como variables categóricas"
)


assert not set(USER_ID_COLUMNS).intersection(
    users_numeric_cols
), (
    "Hay IDs de usuarios tratados como variables numéricas"
)

In [100]:
output_datasets = {
    "bug_reopen_dataset_full": dataset,

    "bug_reopen_modeling_dataset": (
        modeling_dataset
    ),

    "bug_reopen_modeling_dataset_with_users": (
        modeling_dataset_with_users
    ),

    "bug_reopen_text_dataset": text_dataset,

    "bug_reopen_scalar_reconstruction_audit": (
        scalar_reconstruction_audit
    ),

    "bug_reopen_scalar_history_inconsistencies": (
        scalar_history_inconsistencies
    ),

    "bug_reopen_structural_reconstruction_audit": (
        structural_audit
    )
}

for dataset_name, output_df in output_datasets.items():
    parquet_path = OUTPUT_DIR / f"{dataset_name}.parquet"
    csv_path = OUTPUT_DIR / f"{dataset_name}.csv"

    output_df.to_parquet(
        parquet_path,
        index=False
    )

    output_df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"{dataset_name}: "
        f"{output_df.shape[0]} filas, "
        f"{output_df.shape[1]} columnas"
    )

    print(
        "  Parquet:",
        parquet_path
    )

    print(
        "  CSV:",
        csv_path
    )

bug_reopen_dataset_full: 67806 filas, 198 columnas
  Parquet: /app/output/bug_reopen_dataset_full.parquet
  CSV: /app/output/bug_reopen_dataset_full.csv
bug_reopen_modeling_dataset: 67806 filas, 175 columnas
  Parquet: /app/output/bug_reopen_modeling_dataset.parquet
  CSV: /app/output/bug_reopen_modeling_dataset.csv
bug_reopen_modeling_dataset_with_users: 67806 filas, 179 columnas
  Parquet: /app/output/bug_reopen_modeling_dataset_with_users.parquet
  CSV: /app/output/bug_reopen_modeling_dataset_with_users.csv
bug_reopen_text_dataset: 67806 filas, 7 columnas
  Parquet: /app/output/bug_reopen_text_dataset.parquet
  CSV: /app/output/bug_reopen_text_dataset.csv
bug_reopen_scalar_reconstruction_audit: 7 filas, 8 columnas
  Parquet: /app/output/bug_reopen_scalar_reconstruction_audit.parquet
  CSV: /app/output/bug_reopen_scalar_reconstruction_audit.csv
bug_reopen_scalar_history_inconsistencies: 113 filas, 9 columnas
  Parquet: /app/output/bug_reopen_scalar_history_inconsistencies.parquet
  C